# CS-781 Phase 6 — Abstractive Legal Summarisation (notebook J)

Same structure as Phases 2-5: your Phase-1 models + **your** N6 code (full / partial / BitFit / LoRA / LoRMA /
rsLoRA / IA³ / distillation — copied unchanged from Phase 5, plus one new method you write: **prompt tuning**), a
training script, a **generation** script, and a **lab** that turns every generated variant into a Codabench zip plus a
table and chart of tentative scores.

**Generation, not classification.** This phase uses your Phase-1 models **with their own output layer** (no new head).
Decoder / prefix / swa / moe read one sequence `[document context] <s> [summary] </s>` and learn to write the summary
after `<s>` (the loss is on summary tokens only; prefix models see the context bidirectionally). The encoder-decoder
reads the context in its encoder and writes the summary with its decoder. Distillation = token-level: the student
matches the teacher's next-token distribution on every summary token (your Phase-4 `kd_loss`, unchanged).

**Long documents.** A judgment is far longer than the context (~640 tokens for 1024-token models, 768 for encdec).
`train_context` chooses which part a model learns from; at generation time the lab compares **head / salient
sentences (tf-idf) / spread (pieces from across the document)** and optionally **hier** (hierarchical: summarise
chunks, then summarise the joined chunk summaries) — the comparison §12.1 asks for.

**No dev set.** N8 sets aside `VAL_DOCS` training documents (fixed seed) as a validation split: never trained on,
used for the checkpoint choice (summary-token loss) and the **tentative scores** (BERTScore F1 with the same model as
`evaluate_phase6.py`, ROUGE-1 / ROUGE-L with its own code).

**Many zips:** every finished run × `LAB["contexts"]` × `LAB["variants"]` (decoding: greedy / sampling, repetition
controls, length) is one prediction set — e.g. 6 runs × 3 contexts × 4 decodings = 72, of which the 40 best (+ the best
of each run) are zipped. `submissions/index.csv` + `submissions/score_chart.png` list them; **you choose what to
upload**. Settings marked YOUR CHOICE are yours (rule §17.5); declare that the pipeline / generation / lab / chart code
was written with an assistant.

**Kaggle settings:** Accelerator **GPU T4 ×2** · Internet **On** (pip + the BERTScore model) · Add-ons → Secrets →
**`Capstone_Phase_1` attached**.

**Inputs (Add Input)** — the same five as Phase 5:
| Input | Provides |
|---|---|
| `notebook857159535b` | tokenizer (`bpe_tok.py`, `bpe.json`) |
| `notebook27acca3fc3` (notebook D, phase1-continue) | **decoder** checkpoint `continued/best_continued.pt` |
| `Phase1_Training1` | encdec, swa checkpoints |
| dataset `prefix` | prefix checkpoint |
| `Phase1_training2` (phase1-train-extra) | moe (the distillation teacher's base) |
| *optional:* the output of an earlier run of **this** notebook | its runs / generations join the lab; its `p6cache` is reused |

**How to use:** write `method_prompt` in N6 (section 8), then Save Version → Save & Run All (~5 h). N7b stops within
a minute if an input is missing or `method_prompt` is not written yet. Then Output → `submissions/`: `index.csv`,
`score_chart.png` and the zips; download them into your `Phase 6` folder.

## N1. Login, install the metric packages, download the Phase 6 data

In [ ]:
import os, shutil, json, subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rouge-score", "bert-score"], check=True)
from huggingface_hub import login, hf_hub_download
from kaggle_secrets import UserSecretsClient

HF_TOKEN = UserSecretsClient().get_secret("Capstone_Phase_1")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)
os.makedirs("/kaggle/working/p6data", exist_ok=True)
for path in ["Train/train.json", "Test/test.json"]:
    src = hf_hub_download("Exploration-Lab/CS-781-Capstone", f"Phase_6_Summ/{path}", repo_type="dataset", token=HF_TOKEN)
    shutil.copy(src, f"/kaggle/working/p6data/{os.path.basename(path)}")
print({f: f"{os.path.getsize('/kaggle/working/p6data/' + f) / 1e6:.1f} MB" for f in sorted(os.listdir("/kaggle/working/p6data"))})

## N2. GPU check

In [ ]:
import torch
print("GPU count:", torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

## N3. Configuration — experiments, summarisation options, generation variants, lab settings

In [ ]:
import json
# =====================================================================================================
#  N3 — CONFIGURATION. Values marked YOUR CHOICE are yours (course rule §17.5); the comments explain each one.
# =====================================================================================================
#
#  PHASE 6 RULES THIS CONFIG MUST SATISFY (§12)
#  * all FOUR required Phase-1 architectures used: decoder, prefix (hybrid), encdec, swa (efficient);
#    the best four submissions come from DIFFERENT Phase-1 models
#  * ≥ 4 substantially different experiments: ≥ 3 different techniques FROM THE §12.2.1 LIST — Houlsby Adapters,
#    Pfeiffer Adapters, Prefix Tuning, Prompt Tuning, LoRA, QLoRA, AdaLoRA, LORMA, Knowledge Distillation — LORMA
#    COMPULSORY — and ≥ 1 knowledge distillation. Changing only generation settings is NOT a different experiment.
#    The starting plan below covers: LoRMA (encdec_lorma), LoRA (prefix_lora, decoder_lora), Prompt Tuning
#    (swa_prompt — you write it in N6), distillation (kd_moe_to_decoder4). moe_full is the teacher (full fine-tuning
#    is not on the list, but the teacher is part of the distillation experiment).
#  * "handle long documents" — train_context + LAB["contexts"] (head / salient / spread / hier).

CONFIG = {
    "seed": 42,
    "checkpoints": {
        "decoder":   "continued/best_continued.pt",
        "prefix":    "prefix/best.pt",
        "encdec":    "encdec/best.pt",
        "swa":       "swa/best.pt",
        "moe":       "moe/best.pt",
    },
    "val_docs": 200,             # YOUR CHOICE. training documents held out as the validation split (fixed seed)

    # ---------------- defaults for every experiment (an experiment below can override any of them) ----------------
    "defaults": {
        # --- summarisation-specific ---
        "sum_tokens": 384,       # YOUR CHOICE. summary tokens learned / generated at most (encdec: its max_tgt_len
                                 #   384 - 1). The official BERTScore model reads only ~510 of its own tokens of each
                                 #   summary, so much longer summaries add little to the primary metric.
        "ctx_tokens": None,      # context tokens; None = all that fit (1024-token models: 1024 - 2 - sum_tokens
                                 #   - prompt tokens ≈ 638; encdec: 768 - prompt tokens)
        "train_context": "head", # YOUR CHOICE. which part of the document the model learns from: "head" (start),
                                 #   "tail", "headtail", "spread" (8 pieces from across it), "salient" (the highest
                                 #   tf-idf sentences, in document order), or "mix" = a random one of
        "train_context_mix": ["head", "salient", "spread"],   # these for every training example
        "label_smoothing": 0.0,
        # --- compute ---
        "time_budget_min": 60,   # YOUR CHOICE. training minutes per run (or `epochs`, whichever ends first)
        "eval_minutes": 20,      # validation loss every this many minutes (and at each epoch end) on eval_docs
        "eval_docs": 100,        #   validation documents; the best is the checkpoint that is kept
        "epochs": 3,
        # --- the rest as in Phases 2-5 ---
        "lr": 5e-5,              # YOUR CHOICE
        "batch_size": 8,
        "warmup_frac": 0.1,
        "weight_decay": 0.01,
        "grad_clip": 1.0,
        "method_cfg": {},
        "moe_aux_coef": 0.0,
        "train_on_all": False,   # True = train on train + validation (no validation loss)
        "epochs_from": None,
        "teacher": None,
        "kd": {"temperature": 2.0, "alpha": 0.5},
        "student_layers": None,
    },

    # ---------------- the experiments — YOUR CHOICE ----------------
    "experiments": {
        "moe_full":           {"arch": "moe", "method": "full"},                       # the distillation teacher
        "kd_moe_to_decoder4": {"arch": "decoder", "method": "full", "teacher": "moe_full", "student_layers": 4},
        "encdec_lorma":       {"arch": "encdec", "method": "lorma", "method_cfg": {"r": 8, "alpha": 16}, "lr": 5e-4},
        "prefix_lora":        {"arch": "prefix", "method": "lora", "method_cfg": {"r": 8, "alpha": 16}, "lr": 5e-4},
        "swa_prompt":         {"arch": "swa", "method": "prompt", "method_cfg": {"n_tokens": 20}, "lr": 3e-3},
        "decoder_lora":       {"arch": "decoder", "method": "lora", "method_cfg": {"r": 8, "alpha": 16}, "lr": 5e-4},
        # --- optional extras (not in SESSION_EXPERIMENTS below): ---
        "swa_lora":           {"arch": "swa", "method": "lora", "method_cfg": {"r": 8, "alpha": 16}, "lr": 5e-4},
        "prefix_lorma":       {"arch": "prefix", "method": "lorma", "method_cfg": {"r": 8, "alpha": 16}, "lr": 5e-4},
        "decoder_prompt":     {"arch": "decoder", "method": "prompt", "method_cfg": {"n_tokens": 20}, "lr": 3e-3},
        # "name_mix": {<settings of another run>, "train_context": "mix"},       <- long-document strategy in training
    },
}

# ---------------- what THIS session does on the GPUs ----------------
SESSION_EXPERIMENTS = ["moe_full", "prefix_lora", "encdec_lorma", "swa_prompt", "kd_moe_to_decoder4", "decoder_lora"]
SMOKE_EXPERIMENTS = list(SESSION_EXPERIMENTS)
GEN_RUNS = list(SESSION_EXPERIMENTS)   # runs that get generations (a run from an earlier session's output can be added)

# ---------------- generation variants + lab — YOUR CHOICE ----------------
LAB = {
    "val_docs": 150,             # validation documents generated + scored per variant (more = steadier, slower)
    "batch_size": 16,
    "contexts": ["head", "salient", "spread"],   # + "tail", "headtail", "hier" (hierarchical; ~7x slower)
    "hier": {"chunks": 6, "chunk_words": 60},    # hier: chunks per document, words per chunk summary
    "variants": {                # decoding settings (your Phase-1 choices as the starting point)
        "greedy_nr3_w300":  {"temperature": 0.0, "no_repeat_ngram": 3, "min_words": 80, "max_words": 300},
        "greedy_nr3_w180":  {"temperature": 0.0, "no_repeat_ngram": 3, "min_words": 60, "max_words": 180},
        "greedy_rp12_w300": {"temperature": 0.0, "rep_penalty": 1.2, "min_words": 80, "max_words": 300},
        "sample_t07_w300":  {"temperature": 0.7, "top_p": 0.9, "rep_penalty": 1.1, "no_repeat_ngram": 3,
                             "min_words": 80, "max_words": 300},
    },
}
TOP_N_ZIPS = 40              # zips for the 40 best variants (+ the best variant of every run)

CONFIG["lab"] = LAB
json.dump(CONFIG, open("/kaggle/working/run_config_ss.json", "w"), indent=1)
for name, e in CONFIG["experiments"].items():
    assert e["arch"] in CONFIG["checkpoints"], f"{name}: unknown arch {e['arch']}"
    t = e.get("teacher")
    assert not t or t in CONFIG["experiments"], f"{name}: teacher {t} is not an experiment"
for name in SESSION_EXPERIMENTS:
    assert name in CONFIG["experiments"], f"SESSION_EXPERIMENTS: {name} is not defined"
budget = [CONFIG["experiments"][e].get("time_budget_min", CONFIG["defaults"]["time_budget_min"]) for e in SESSION_EXPERIMENTS]
n_var = len(LAB["contexts"]) * len(LAB["variants"])
print(f"{len(CONFIG['experiments'])} experiments defined; this session trains {len(SESSION_EXPERIMENTS)}: {SESSION_EXPERIMENTS}")
print(f"generation: {len(GEN_RUNS)} runs x {n_var} variants = {len(GEN_RUNS) * n_var} prediction sets")
print(f"rough GPU time: ~{(sum(b + 5 for b in budget) + 15 * len(GEN_RUNS) * n_var / 12) / 2 / 60:.1f} h on T4 x2 "
      "(training + generation; + ~15 min scoring)")

## N4. Pipeline code: data + validation split, tokenisation cache, context selection, examples, cached generation, files

In [ ]:
%%writefile ss_common.py
"""Phase 6 (abstractive summarisation) pipeline: data + a held-out validation split carved from train, one-time
Phase-1 tokenisation (documents sentence by sentence, summaries whole; cached on disk), context selection for long
documents (head / tail / head+tail / spread / salient sentences), training examples for your two model formats
(language model: [document context] <s> [summary] </s>; encoder-decoder: source = context, target = summary),
fast generation with a key/value cache, model wrappers, run finder, files."""
import glob, json, os, re, sys
from multiprocessing import Pool
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from torch.nn.utils import parametrize

INPUT_ROOT = os.environ.get("P6_INPUT", "/kaggle/input")
WORK = os.environ.get("P6_WORK", "/kaggle/working")
DATA = f"{WORK}/p6data"
CACHE = f"{WORK}/p6cache"
SPLITS = ("train", "val", "test")
IGNORE = -100
LEN_BUCKET = 32          # a context shorter than the budget is cut to a multiple of this (fewer distinct lengths)

def find_input(pattern):
    hits = sorted(glob.glob(f"{INPUT_ROOT}/**/{pattern}", recursive=True))
    if not hits:
        raise FileNotFoundError(f"'{pattern}' not found under {INPUT_ROOT} — is the right notebook/dataset added as input?")
    return hits[0]

sys.path.insert(0, os.path.dirname(find_input("bpe_tok.py")))
import bpe_tok

_BPE = None
def bpe_tok_bpe():
    global _BPE
    if _BPE is None:
        _BPE = bpe_tok.load_bpe(find_input("bpe.json"))
    return _BPE

_bpe = bpe_tok_bpe()
PAD, UNK, BOS, EOS = (_bpe.stoi[s] for s in ("<pad>", "<unk>", "<s>", "</s>"))
SPECIAL = {PAD, UNK, BOS, EOS}
WORD_MARK = "".join(chr(b) for b in "▁".encode("utf-8"))
WORD_START = torch.tensor([t.startswith(WORD_MARK) for t in _bpe.itos])

def decode(ids):
    """token ids -> text (special tokens dropped)"""
    return bpe_tok.detokenize([_bpe.itos[i] for i in ids if i not in SPECIAL])

# ------------------------------------------------------------------------------------------------ data
def _text(x):
    if isinstance(x, list):
        return " ".join(map(str, x))
    return x if isinstance(x, str) else None

def read_split(name):
    """-> list of {"id": str, "document": str, "summary": str or None}. Accepts the documented list of
    {"id", "document", "summary"} and close variants (a dict keyed by id; "text" / "judgment" instead of "document";
    "abstract" / "reference" instead of "summary"; text given as a list of sentences)."""
    obj = json.load(open(f"{DATA}/{name}.json", encoding="utf-8"))
    if isinstance(obj, dict) and isinstance(obj.get("data"), list):
        obj = obj["data"]
    items = list(obj.items()) if isinstance(obj, dict) else [(None, x) for x in obj]
    out = []
    for k, it in items:
        if isinstance(it, str):
            it = {"document": it}
        doc_id = it.get("id", it.get("doc_id", it.get("case_id", k)))
        doc = _text(it.get("document", it.get("text", it.get("judgment"))))
        if doc_id is None or doc is None:
            raise ValueError(f"{name}.json: cannot read an example (keys: {list(it)[:8]}). Expected "
                             '{"id": ..., "document": "...", "summary": "..."}')
        summ = _text(it.get("summary", it.get("abstract", it.get("reference"))))
        out.append({"id": str(doc_id), "document": doc, "summary": summ})
    return out

_SENT = re.compile(r"(?<=[.!?;:])\s+(?=[\"'(\[A-Z0-9])|\n+")

def sentences(text):
    return [s for s in (x.strip() for x in _SENT.split(text)) if s]

# ------------------------------------------------------------------------------------------------ tokenisation cache
_ENC = None
def _init_worker():
    global _ENC
    _ENC = bpe_tok.FastEncoder(bpe_tok_bpe())

def _encode(d):
    sents = [np.asarray(_ENC.encode(s), dtype=np.int32) for s in sentences(d["document"])]
    sents = [s for s in sents if len(s)] or [np.array([UNK], dtype=np.int32)]
    summ = np.asarray(_ENC.encode(d["summary"]), dtype=np.int32) if d["summary"] else np.zeros(0, np.int32)
    return sents, summ

def build_cache(val_docs, seed=0, workers=None, limit=None):
    """Tokenises every document ONCE with your Phase-1 tokenizer (bpe_tok.FastEncoder, as in pretraining), sentence by
    sentence (so contexts can be built from whole sentences), and every summary whole. The validation split = val_docs
    training documents chosen at random (fixed seed) — Phase 6 has no public dev set. Per split: document tokens
    (doc_ids.npy) + sentence boundaries (sent_off.npy) + first sentence of each document (doc_sent.npy) + summary
    tokens (sum_ids.npy, sum_off.npy) + ids and reference summaries (meta.json)."""
    os.makedirs(CACHE, exist_ok=True)
    if all(os.path.exists(f"{CACHE}/{s}_meta.json") for s in SPLITS):
        m = json.load(open(f"{CACHE}/val_meta.json"))
        if len(m["ids"]) != val_docs:
            print(f"NOTE: the cache has a validation split of {len(m['ids'])} documents, not {val_docs}; delete "
                  f"{CACHE} to rebuild it")
        return
    train = read_split("train")[:limit]
    test = read_split("test")[:limit]
    order = np.random.RandomState(seed).permutation(len(train))
    val_set = set(order[:val_docs].tolist())
    parts = {"train": [d for i, d in enumerate(train) if i not in val_set],
             "val": [train[i] for i in sorted(val_set)], "test": test}
    dtype = np.uint16 if len(bpe_tok_bpe().itos) <= 65535 else np.int32
    with Pool(workers or os.cpu_count(), initializer=_init_worker) as pool:
        for split, docs in parts.items():
            enc = pool.map(_encode, docs, chunksize=8)
            sent_lens = [len(s) for sents, _ in enc for s in sents]
            sent_off = np.zeros(len(sent_lens) + 1, dtype=np.int64); sent_off[1:] = np.cumsum(sent_lens)
            doc_sent = np.zeros(len(enc) + 1, dtype=np.int64); doc_sent[1:] = np.cumsum([len(s) for s, _ in enc])
            sum_off = np.zeros(len(enc) + 1, dtype=np.int64); sum_off[1:] = np.cumsum([len(x) for _, x in enc])
            flat = [s for sents, _ in enc for s in sents]
            np.save(f"{CACHE}/{split}_doc_ids.npy", np.concatenate(flat).astype(dtype) if flat else np.zeros(0, dtype))
            np.save(f"{CACHE}/{split}_sent_off.npy", sent_off)
            np.save(f"{CACHE}/{split}_doc_sent.npy", doc_sent)
            sums = [x for _, x in enc]
            np.save(f"{CACHE}/{split}_sum_ids.npy", np.concatenate(sums).astype(dtype) if sums else np.zeros(0, dtype))
            np.save(f"{CACHE}/{split}_sum_off.npy", sum_off)
            meta = {"ids": [d["id"] for d in docs],
                    "refs": [d["summary"] for d in docs] if split == "val" else None}
            json.dump(meta, open(f"{CACHE}/{split}_meta.json", "w"))
            print(f"{split}: {len(docs)} documents, {sent_off[-1]:,} document tokens, {sum_off[-1]:,} summary tokens cached")
    # inverse document frequency of every token over the training documents (for "salient" context selection)
    tr = Split("train")
    df = np.zeros(len(bpe_tok_bpe().itos), dtype=np.int64)
    for i in range(len(tr)):
        df[np.unique(tr.tokens(i))] += 1
    np.save(f"{CACHE}/idf.npy", np.log((1 + len(tr)) / (1 + df)).astype(np.float32))

class Split:
    """one cached split: ids, reference summaries (val), document tokens / sentence spans / summary tokens of i"""
    def __init__(self, name):
        m = json.load(open(f"{CACHE}/{name}_meta.json"))
        self.name, self.ids, self.refs = name, m["ids"], m["refs"]
        self.flat = np.load(f"{CACHE}/{name}_doc_ids.npy", mmap_mode="r")
        self.sent_off = np.load(f"{CACHE}/{name}_sent_off.npy")
        self.doc_sent = np.load(f"{CACHE}/{name}_doc_sent.npy")
        self.sflat = np.load(f"{CACHE}/{name}_sum_ids.npy", mmap_mode="r")
        self.sum_off = np.load(f"{CACHE}/{name}_sum_off.npy")
    def __len__(self):
        return len(self.ids)
    def _span(self, i):
        return self.sent_off[self.doc_sent[i]], self.sent_off[self.doc_sent[i + 1]]
    def tokens(self, i):
        a, b = self._span(i)
        return np.asarray(self.flat[a:b], dtype=np.int32)
    def sents(self, i):
        """sentence boundaries of document i, relative to its first token: [(start, end), ...]"""
        o = self.sent_off[self.doc_sent[i]:self.doc_sent[i + 1] + 1] - self.sent_off[self.doc_sent[i]]
        return list(zip(o[:-1].tolist(), o[1:].tolist()))
    def summary(self, i):
        return np.asarray(self.sflat[self.sum_off[i]:self.sum_off[i + 1]], dtype=np.int32)
    def doc_lengths(self):
        return np.diff(self.sent_off[self.doc_sent])
    def sum_lengths(self):
        return np.diff(self.sum_off)

_IDF = None
def idf():
    global _IDF
    if _IDF is None:
        _IDF = np.load(f"{CACHE}/idf.npy")
    return _IDF

# ------------------------------------------------------------------------------------------------ context selection
CONTEXTS = ("head", "tail", "headtail", "spread", "salient")

def select_context(split, i, budget, how, spread_parts=8):
    """the part of document i (at most `budget` tokens) the model sees:
    head      its first tokens                      tail      its last tokens
    headtail  first half + last half of the budget  spread    spread_parts equal pieces from across the document,
                                                              each starting at a sentence start
    salient   the most informative sentences (mean tf-idf of their tokens within the document), in document order"""
    t = split.tokens(i)
    if len(t) <= budget:
        out = t
    elif how == "head":
        out = t[:budget]
    elif how == "tail":
        out = t[-budget:]
    elif how == "headtail":
        out = np.concatenate([t[:budget // 2], t[len(t) - (budget - budget // 2):]])
    elif how == "spread":
        starts = np.array([s for s, _ in split.sents(i)])
        piece = budget // spread_parts
        out = []
        for k in range(spread_parts):
            want = int(k * (len(t) - piece) / max(1, spread_parts - 1))
            s = int(starts[np.searchsorted(starts, want, side="right") - 1])
            out.append(t[s:s + piece])
        out = np.concatenate(out)[:budget]
    elif how == "salient":
        sp = split.sents(i)
        w = idf()[t] * (np.bincount(t, minlength=len(idf()))[t] / len(t))   # tf-idf of every token in this document
        score = np.array([w[a:b].mean() for a, b in sp])
        keep, used = [], 0
        for k in np.argsort(-score):
            if used >= budget:
                break
            keep.append(k); used += sp[k][1] - sp[k][0]
        out = np.concatenate([t[sp[k][0]:sp[k][1]] for k in sorted(keep)])[:budget]
    else:
        raise ValueError(f"context must be one of {CONTEXTS}, got {how!r}")
    if len(out) < budget and len(out) >= LEN_BUCKET:
        out = out[:len(out) - len(out) % LEN_BUCKET]
    return np.asarray(out, dtype=np.int32)

# ------------------------------------------------------------------------------------------------ model wrappers
class LMHolder(nn.Module):
    """your Phase-1 model as the 'backbone' your N6 methods expect (they use backbone.lm and backbone.arch); the LM
    keeps its output layer — summarisation generates text"""
    def __init__(self, arch, lm):
        super().__init__()
        self.arch, self.lm = arch, lm

class SumModel(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone
    def forward(self, *a, **k):
        return self.backbone.lm(*a, **k)

def fmt_of(lm):
    """"ed" = encoder-decoder (source / target); "lm" = one sequence [context] <s> [summary] (decoder, prefix, swa, moe)"""
    if lm.kind == "encdec":
        return "ed"
    if lm.kind in ("causal", "prefix"):
        return "lm"
    raise ValueError(f"a '{lm.kind}' model cannot generate left to right — use decoder / prefix / encdec / swa / moe")

TINY = {   # used only by the CPU smoke test (random weights, no checkpoint)
    "decoder":   dict(d_model=32, n_layers=2, n_heads=4, max_len=256, dropout=0.0),
    "prefix":    dict(d_model=32, n_layers=2, n_heads=4, max_len=256, dropout=0.0),
    "encdec":    dict(d_model=32, n_enc_layers=2, n_dec_layers=2, n_heads=4, max_len=192, max_tgt_len=64, dropout=0.0),
    "swa":       dict(d_model=32, n_layers=2, n_heads=4, window=16, max_len=256, dropout=0.0),
    "diffusion": dict(d_model=32, n_layers=2, n_heads=4, max_len=256, dropout=0.0),
    "moe":       dict(d_model=32, n_layers=2, n_heads=4, d_ff=64, n_experts=4, max_len=256, dropout=0.0),
}

def build_lm(arch, cfg):
    import models, models_x
    return (models_x if arch in ("diffusion", "moe") else models).build_model(arch, vocab_size=len(bpe_tok_bpe().itos), **cfg)

def load_holder(arch, ckpt_pattern, smoke=False):
    if smoke:
        lm = build_lm(arch, TINY[arch])
    else:
        ck = torch.load(find_input(ckpt_pattern), map_location="cpu", weights_only=False)
        assert ck.get("arch", arch) == arch, f"checkpoint {ckpt_pattern} is a '{ck.get('arch')}' model, not '{arch}'"
        lm = build_lm(arch, ck["cfg"])
        lm.load_state_dict(ck["model"])
    return LMHolder(arch, lm)

def iter_adaptable(module, skip=("head",)):
    for name, mod in module.named_modules():
        if any(s in name.split(".") for s in skip):
            continue
        if isinstance(mod, nn.MultiheadAttention) and getattr(mod, "in_proj_weight", None) is not None:
            yield name, mod, "in_proj_weight", mod.in_proj_weight
        elif isinstance(mod, nn.Linear):
            yield name, mod, "weight", mod.weight

def layer_stack(backbone):
    lm = backbone.lm
    if backbone.arch == "encdec":
        return "enc_layers", lm.enc_layers, lm.enc_norm
    if hasattr(lm, "blocks"):
        return "blocks", lm.blocks, lm.norm
    return "layers", lm.layers, lm.norm

def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    return total, sum(p.numel() for p in model.parameters() if p.requires_grad)

def merge_adapters(model):
    """bakes every parametrisation (LoRA, LoRMA, IA3, ...) into its weight — same outputs, faster generation"""
    for mod in model.modules():
        if parametrize.is_parametrized(mod):
            for name in list(mod.parametrizations.keys()):
                parametrize.remove_parametrizations(mod, name, leave_parametrized=True)

# ------------------------------------------------------------------------------------------------ budgets + examples
def budgets(lm, n_prompt, sum_tokens, ctx_tokens=None):
    """-> (context tokens, summary tokens) that fit the model: lm format [prompt][context]<s>[summary]</s> within
    max_len; ed format source [prompt][context] within max_len, target <s>[summary] within max_tgt_len"""
    if fmt_of(lm) == "ed":
        S = min(sum_tokens, lm.max_tgt_len - 1)
        C = lm.max_len - n_prompt
    else:
        S = sum_tokens
        C = lm.max_len - n_prompt - 2 - S
    assert C >= 64, f"no room for a context: max_len {lm.max_len}, summary {S}, prompt {n_prompt}"
    return (min(C, ctx_tokens) if ctx_tokens else C), S

def lm_example(ctx, summ, S, n_prompt):
    """[pad]*n_prompt + context + <s> + summary[:S] (+ </s> if the summary was not cut) -> (input, target, P);
    the loss is on summary tokens (and </s>) only; P = length of the part before the summary (prefix models see it
    bidirectionally)"""
    s = list(summ[:S]) + ([EOS] if len(summ) <= S else [])
    seq = [PAD] * n_prompt + list(ctx) + [BOS] + s
    P = n_prompt + len(ctx) + 1
    return seq[:-1], [IGNORE] * (P - 1) + seq[P:], P

def ed_example(ctx, summ, S, n_prompt):
    """source = [pad]*n_prompt + context; decoder input = <s> + target[:-1]; target = summary[:S] (+ </s>)"""
    tgt = list(summ[:S]) + ([EOS] if len(summ) <= S else [])
    return [PAD] * n_prompt + list(ctx), [BOS] + tgt[:-1], tgt

def pad_to(rows, value):
    n = max(len(r) for r in rows)
    return [list(r) + [value] * (n - len(r)) for r in rows]

def group_batches(keys, batch_size, shuffle, rng):
    """indices grouped by key (equal context length: your models have no padding mask), batch_size per batch"""
    by = {}
    for j, k in enumerate(keys):
        by.setdefault(k, []).append(j)
    out = []
    for _, js in by.items():
        if shuffle:
            js = [js[x] for x in rng.permutation(len(js))]
        out += [js[a:a + batch_size] for a in range(0, len(js), batch_size)]
    if shuffle:
        out = [out[x] for x in rng.permutation(len(out))]
    return out

def seq_logits(model, fmt, batch, device):
    """batch = list of examples of ONE context length -> (logits, targets) for the summary-token loss"""
    if fmt == "lm":
        P = batch[0][2]
        x = torch.tensor(pad_to([b[0] for b in batch], PAD), dtype=torch.long, device=device)
        y = torch.tensor(pad_to([b[1] for b in batch], IGNORE), dtype=torch.long, device=device)
        lm = model.backbone.lm
        logits = lm(x, prefix_len=P) if lm.kind == "prefix" else lm(x)
    else:
        src = torch.tensor([b[0] for b in batch], dtype=torch.long, device=device)
        tin = torch.tensor(pad_to([b[1] for b in batch], PAD), dtype=torch.long, device=device)
        y = torch.tensor(pad_to([b[2] for b in batch], IGNORE), dtype=torch.long, device=device)
        logits = model.backbone.lm(src, tin)
    return logits, y

# ------------------------------------------------------------------------------------------------ generation
class CachedLM:
    """Incremental forward pass for your decoder / prefix / swa / moe models: keeps every layer's keys and values, so
    each new token attends to the stored ones instead of re-running the whole sequence. Same arithmetic as model(ids)
    (checked against it by the CPU smoke test), many times faster. Prefix models: the prompt is the prefix (seen
    bidirectionally), as in training. swa: ALiBi + sliding window bias rebuilt for each new position."""
    def __init__(self, lm):
        self.lm = lm
        self.blocks = getattr(lm, "blocks", None)
        self.layers = self.blocks if self.blocks is not None else lm.layers
        self.alibi = not hasattr(lm, "pos_emb")
        if self.blocks is None:
            assert all(l.norm_first for l in lm.layers), "CachedLM expects norm_first=True layers"
        if self.alibi:
            H = lm.n_heads
            self.slopes = torch.tensor([2 ** (-8 * (h + 1) / H) for h in range(H)])

    def _mask(self, start, T, S, device, dtype):
        if self.alibi:
            q = torch.arange(start, start + T, device=device)[:, None]
            k = torch.arange(S, device=device)[None, :]
            dist = (q - k).float()
            bias = -self.slopes.to(device)[:, None, None] * dist
            bias = bias.masked_fill((dist < 0) | (dist >= self.lm.window), float("-inf"))
            return bias[None].to(dtype), False
        if T == 1 or self.lm.kind == "prefix":
            return None, False
        return None, True

    def _attn(self, mha, h, i, start):
        B, T, D = h.shape
        H = mha.num_heads
        q, k, v = F.linear(h, mha.in_proj_weight, mha.in_proj_bias).chunk(3, dim=-1)
        q, k, v = (t.view(B, T, H, D // H).transpose(1, 2) for t in (q, k, v))
        if start == 0:
            self.k.append(k); self.v.append(v)
        else:
            self.k[i] = torch.cat([self.k[i], k], 2); self.v[i] = torch.cat([self.v[i], v], 2)
        mask, causal = self._mask(start, T, self.k[i].size(2), h.device, q.dtype)
        o = F.scaled_dot_product_attention(q, self.k[i], self.v[i], attn_mask=mask, is_causal=causal)
        return mha.out_proj(o.transpose(1, 2).reshape(B, T, D))

    def _run(self, idx, start):
        lm = self.lm
        x = lm.tok_emb(idx)
        if not self.alibi:
            x = x + lm.pos_emb(torch.arange(start, start + idx.size(1), device=idx.device))
        for i, l in enumerate(self.layers):
            if self.blocks is not None:
                x = x + l.drop(self._attn(l.attn, l.norm1(x), i, start))
                x = x + l.moe(l.norm2(x))
            else:
                x = x + l.dropout1(self._attn(l.self_attn, l.norm1(x), i, start))
                x = x + l.dropout2(l.linear2(l.dropout(l.activation(l.linear1(l.norm2(x))))))
        return lm.head(lm.norm(x[:, -1]))

    def prefill(self, idx):
        self.k, self.v, self.t = [], [], idx.size(1)
        return self._run(idx, 0)

    def step(self, last):
        out = self._run(last, self.t)
        self.t += 1
        return out

class EncDecRunner:
    """encoder run once per batch; the decoder re-reads the (short) target each step"""
    def __init__(self, lm):
        self.lm = lm
    def prefill(self, src):
        import models
        lm = self.lm
        h = lm.tok_emb(src) + lm.src_pos_emb(torch.arange(src.size(1), device=src.device))
        for layer in lm.enc_layers:
            h = layer(h)
        self.mem = lm.enc_norm(h)
        self.tgt = torch.full((src.size(0), 1), BOS, dtype=torch.long, device=src.device)
        return self._dec()
    def step(self, last):
        self.tgt = torch.cat([self.tgt, last], 1)
        return self._dec()
    def _dec(self):
        import models
        lm, U = self.lm, self.tgt.size(1)
        y = lm.tok_emb(self.tgt) + lm.tgt_pos_emb(torch.arange(U, device=self.tgt.device))
        mask = models.causal_mask(U, self.tgt.device)
        for layer in lm.dec_layers:
            y = layer(y, self.mem, tgt_mask=mask)
        return lm.head(lm.dec_norm(y[:, -1]))

def sample_next(logits, generated, temperature, top_k, top_p, rep_penalty):
    """your Phase-1 sampling: repetition penalty, then greedy (temperature 0) or temperature + top-k + top-p"""
    if rep_penalty != 1.0 and generated.size(1) > 0:
        prev = torch.gather(logits, 1, generated)
        logits.scatter_(1, generated, torch.where(prev > 0, prev / rep_penalty, prev * rep_penalty))
    if temperature <= 0:
        return logits.argmax(-1)
    logits = logits / temperature
    if top_k and top_k < logits.size(-1):
        kth = torch.topk(logits, top_k).values[:, -1:]
        logits = logits.masked_fill(logits < kth, -float("inf"))
    if top_p < 1.0:
        sorted_logits, idx = torch.sort(logits, descending=True)
        probs = sorted_logits.softmax(-1)
        sorted_logits = sorted_logits.masked_fill(probs.cumsum(-1) - probs > top_p, -float("inf"))
        logits = torch.full_like(logits, -float("inf")).scatter(1, idx, sorted_logits)
    return torch.multinomial(logits.softmax(-1), 1).squeeze(1)

@torch.no_grad()
def generate(model, contexts, n_prompt, max_new, device, temperature=0.0, top_k=0, top_p=1.0, rep_penalty=1.0,
             no_repeat_ngram=0, min_words=0, max_words=300, batch_size=16, seed=0):
    """contexts: list of token arrays (one per document). Each becomes the model's input ([pad]*n_prompt + context
    (+ <s> for lm models)); the summary is generated until </s>, max_words words or max_new tokens. </s> is not
    allowed before min_words words. no_repeat_ngram = n > 0: a token may not complete an n-gram the summary already
    contains. -> list of summary texts"""
    torch.manual_seed(seed)
    model.eval()
    lm = model.backbone.lm
    fmt = fmt_of(lm)
    ws = WORD_START.to(device)
    banned = torch.tensor([PAD, UNK, BOS], device=device)
    n = int(no_repeat_ngram or 0)
    out = [""] * len(contexts)
    for b in group_batches([len(c) for c in contexts], batch_size, False, None):
        rows = [[PAD] * n_prompt + list(contexts[j]) + ([BOS] if fmt == "lm" else []) for j in b]
        x = torch.tensor(rows, dtype=torch.long, device=device)
        B = len(b)
        runner = CachedLM(lm) if fmt == "lm" else EncDecRunner(lm)
        gen = torch.empty(B, 0, dtype=torch.long, device=device)
        done = torch.zeros(B, dtype=torch.bool, device=device)
        seen, hist = [dict() for _ in range(B)], [[] for _ in range(B)]
        steps = min(max_new, lm.max_len - x.size(1)) if fmt == "lm" else min(max_new, lm.max_tgt_len - 1)
        for step in range(steps):
            with torch.autocast("cuda", dtype=torch.float16, enabled=str(device).startswith("cuda")):
                logits = (runner.prefill(x) if step == 0 else runner.step(gen[:, -1:])).float()
            logits[:, banned] = -float("inf")
            words = ws[gen].sum(1) if gen.size(1) else torch.zeros(B, dtype=torch.long, device=device)
            logits[words < min_words, EOS] = -float("inf")
            if n > 0 and gen.size(1) >= n - 1:
                for j in range(B):
                    nb = seen[j].get(tuple(hist[j][-(n - 1):]) if n > 1 else ())
                    if nb:
                        logits[j, list(nb)] = -float("inf")
            nxt = sample_next(logits, gen, temperature, top_k, top_p, rep_penalty)
            nxt = torch.where(done, torch.full_like(nxt, EOS), nxt)
            gen = torch.cat([gen, nxt[:, None]], 1)
            if n > 0:
                for j, t in enumerate(nxt.tolist()):
                    hist[j].append(t)
                    if len(hist[j]) >= n:
                        seen[j].setdefault(tuple(hist[j][-n:-1]), set()).add(t)
            done |= (nxt == EOS) | (ws[gen].sum(1) > max_words)
            if done.all():
                break
        for r, j in enumerate(b):
            ids = gen[r].tolist()
            ids = ids[:ids.index(EOS)] if EOS in ids else ids
            out[j] = " ".join(decode(ids).split()[:max_words])
    return out

# ------------------------------------------------------------------------------------------------ runs + files
def find_runs(filename, sub="ss_runs", overrides=None):
    """{experiment: folder} for every '.../<sub>/<experiment>/<filename>' under /kaggle/working and /kaggle/input.
    Found more than once: /kaggle/working wins, then the newest file."""
    hits = {}
    for rank, root in enumerate((WORK, INPUT_ROOT)):
        for p in glob.glob(f"{root}/**/{sub}/*/{filename}", recursive=True):
            folder = os.path.dirname(p)
            exp, key = os.path.basename(folder), (rank, -os.path.getmtime(p))
            if exp not in hits or key < hits[exp][0]:
                hits[exp] = (key, folder)
    out = {e: f for e, (_, f) in sorted(hits.items())}
    out.update(overrides or {})
    return out

def write_submission(ids, summaries, path_json, path_zip=None):
    """official list format: [{"id": "...", "summary": "..."}, ...], one non-empty summary per test document"""
    rows = [{"id": i, "summary": summaries[i].strip() or "the"} for i in ids]
    json.dump(rows, open(path_json, "w", encoding="utf-8"), ensure_ascii=False)
    if path_zip:
        import zipfile
        with zipfile.ZipFile(path_zip, "w", zipfile.ZIP_DEFLATED) as z:
            z.write(path_json, arcname="prediction.json")

## N5. Your Phase-1 model code (unchanged copies of your `models.py` and `models_x.py`)

In [ ]:
%%writefile models.py
# =====================================================================================================
#  CS-781 Phase 1 — four language models                      
# =====================================================================================================

import math
import torch
import torch.nn as nn


# =====================================================================================================
#  PART 1 — MASKS
# =====================================================================================================

def causal_mask(T, device):
    mask = torch.ones(T, T, dtype=torch.bool, device=device).triu(diagonal=1)
    return mask


def prefix_mask(T, P, device):
    mask = causal_mask(T, device)
    mask[:, :P] = False
    return mask


def alibi_window_bias(T, n_heads, window, device):
    i = torch.arange(T, device=device)[:, None]
    j = torch.arange(T, device=device)[None, :]

    dist = i - j

    slopes = torch.tensor(
        [2 ** (-8 * (h + 1) / n_heads) for h in range(n_heads)],
        device=device,
        dtype=torch.float32,
    )[:, None, None]

    bias = -slopes * dist

    bias = bias.masked_fill(
        (dist < 0) | (dist >= window),
        float("-inf")
    )

    return bias


# =====================================================================================================
#  PART 2 — SHARED HELPERS
# =====================================================================================================

def make_layers(n_layers, d_model, n_heads, dropout):

    return nn.ModuleList([
        nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=4 * d_model,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        for _ in range(n_layers)
    ])


def init_weights(module):
    if isinstance(module, (nn.Linear, nn.Embedding)):
        nn.init.normal_(module.weight, mean=0.0, std=0.02)

    if isinstance(module, nn.Linear) and module.bias is not None:
        nn.init.zeros_(module.bias)


# =====================================================================================================
#  MODEL 1 — DECODER-ONLY (GPT-style)
# =====================================================================================================

class DecoderLM(nn.Module):
    kind = "causal"

    def __init__(self, vocab_size, d_model, n_layers, n_heads, max_len, dropout=0.1):
        super().__init__()

        self.max_len = max_len

        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)

        self.layers = make_layers(
            n_layers,
            d_model,
            n_heads,
            dropout
        )

        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)

        self.apply(init_weights)

        # Weight tying
        self.head.weight = self.tok_emb.weight

    def forward(self, idx):
        B, T = idx.shape

        pos = torch.arange(T, device=idx.device)

        x = self.tok_emb(idx) + self.pos_emb(pos)

        mask = causal_mask(T, idx.device)

        for layer in self.layers:
            x = layer(x, src_mask=mask)

        return self.head(self.norm(x))


# =====================================================================================================
#  MODEL 2 — PREFIX-LM
# =====================================================================================================

class PrefixLM(nn.Module):
    kind = "prefix"

    def __init__(self, vocab_size, d_model, n_layers, n_heads, max_len, dropout=0.1):
        super().__init__()

        self.max_len = max_len

        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)

        self.layers = make_layers(
            n_layers,
            d_model,
            n_heads,
            dropout
        )

        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)

        self.apply(init_weights)

        # Weight tying
        self.head.weight = self.tok_emb.weight

    def forward(self, idx, prefix_len):
        B, T = idx.shape

        pos = torch.arange(T, device=idx.device)

        x = self.tok_emb(idx) + self.pos_emb(pos)

        mask = prefix_mask(T, prefix_len, idx.device)

        for layer in self.layers:
            x = layer(x, src_mask=mask)

        return self.head(self.norm(x))


# =====================================================================================================
#  MODEL 3 — ENCODER-DECODER
# =====================================================================================================

class EncDecLM(nn.Module):
    kind = "encdec"

    def __init__(
        self,
        vocab_size,
        d_model,
        n_enc_layers,
        n_dec_layers,
        n_heads,
        max_len,
        max_tgt_len,
        dropout=0.1,
    ):
        super().__init__()

        self.max_len, self.max_tgt_len = max_len, max_tgt_len

        # Shared token embedding
        self.tok_emb = nn.Embedding(vocab_size, d_model)

        # Separate position embeddings
        self.src_pos_emb = nn.Embedding(max_len, d_model)
        self.tgt_pos_emb = nn.Embedding(max_tgt_len, d_model)

        # Encoder
        self.enc_layers = make_layers(
            n_enc_layers,
            d_model,
            n_heads,
            dropout
        )

        self.enc_norm = nn.LayerNorm(d_model)

        # Decoder
        self.dec_layers = nn.ModuleList([
            nn.TransformerDecoderLayer(
                d_model=d_model,
                nhead=n_heads,
                dim_feedforward=4 * d_model,
                dropout=dropout,
                activation="gelu",
                batch_first=True,
                norm_first=True,
            )
            for _ in range(n_dec_layers)
        ])

        self.dec_norm = nn.LayerNorm(d_model)

        self.head = nn.Linear(d_model, vocab_size, bias=False)

        self.apply(init_weights)

        # Weight tying
        self.head.weight = self.tok_emb.weight

    def forward(self, src, tgt_in):
        S, U = src.size(1), tgt_in.size(1)

        # -------------------------
        # ENCODER
        # -------------------------

        src_pos = torch.arange(S, device=src.device)

        mem = self.tok_emb(src) + self.src_pos_emb(src_pos)

        for layer in self.enc_layers:
            mem = layer(mem)

        mem = self.enc_norm(mem)

        # -------------------------
        # DECODER
        # -------------------------

        tgt_pos = torch.arange(U, device=tgt_in.device)

        y = self.tok_emb(tgt_in) + self.tgt_pos_emb(tgt_pos)

        tgt_mask = causal_mask(U, tgt_in.device)

        for layer in self.dec_layers:
            y = layer(
                y,
                mem,
                tgt_mask=tgt_mask
            )

        return self.head(self.dec_norm(y))


# =====================================================================================================
#  MODEL 4 — EFFICIENT: SLIDING-WINDOW ATTENTION + ALiBi
# =====================================================================================================

class SlidingWindowLM(nn.Module):
    kind = "causal"

    def __init__(
        self,
        vocab_size,
        d_model,
        n_layers,
        n_heads,
        window,
        max_len,
        dropout=0.1,
    ):
        super().__init__()

        self.max_len, self.n_heads, self.window = max_len, n_heads, window

        self.tok_emb = nn.Embedding(vocab_size, d_model)

        self.layers = make_layers(
            n_layers,
            d_model,
            n_heads,
            dropout
        )

        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)

        self.apply(init_weights)

        # Weight tying
        self.head.weight = self.tok_emb.weight

    def forward(self, idx):
        B, T = idx.shape

        # Token embeddings only
        x = self.tok_emb(idx)

        # ALiBi + sliding-window bias
        bias = alibi_window_bias(
            T,
            self.n_heads,
            self.window,
            idx.device
        )

        # (n_heads, T, T)
        # -> (B * n_heads, T, T)
        bias = bias.repeat(B, 1, 1)

        for layer in self.layers:
            x = layer(
                x,
                src_mask=bias
            )

        return self.head(self.norm(x))


# =====================================================================================================
#  FACTORY — do not change.
# =====================================================================================================

def build_model(arch, vocab_size, **cfg):
    classes = {
        "decoder": DecoderLM,
        "prefix": PrefixLM,
        "encdec": EncDecLM,
        "swa": SlidingWindowLM
    }

    return classes[arch](vocab_size, **cfg)

In [ ]:
%%writefile models_x.py

# =====================================================================================================
# CS-781 Phase 1 — two extra architectures
# =====================================================================================================

import math
import torch
import torch.nn as nn


# =====================================================================================================
# PART 0 — HELPERS
# =====================================================================================================

def causal_mask(T, device):
    return torch.ones(
        T, T,
        dtype=torch.bool,
        device=device
    ).triu(diagonal=1)


def make_layers(n_layers, d_model, n_heads, dropout):
    return nn.ModuleList([
        nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=4 * d_model,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        for _ in range(n_layers)
    ])


def init_weights(module):
    if isinstance(module, (nn.Linear, nn.Embedding)):
        nn.init.normal_(
            module.weight,
            mean=0.0,
            std=0.02
        )

    if isinstance(module, nn.Linear) and module.bias is not None:
        nn.init.zeros_(module.bias)


# =====================================================================================================
# MODEL 5 — LexDiffusionLM
# =====================================================================================================

class LexDiffusionLM(nn.Module):
    kind = "diffusion"

    def __init__(
        self,
        vocab_size,
        d_model,
        n_layers,
        n_heads,
        max_len,
        dropout=0.1
    ):
        super().__init__()

        self.max_len = max_len

        # Token embedding
        self.tok_emb = nn.Embedding(
            vocab_size,
            d_model
        )

        # Learned positional embedding
        self.pos_emb = nn.Embedding(
            max_len,
            d_model
        )

        # Transformer layers
        self.layers = make_layers(
            n_layers,
            d_model,
            n_heads,
            dropout
        )

        # Final normalization
        self.norm = nn.LayerNorm(d_model)

        # Output head
        self.head = nn.Linear(
            d_model,
            vocab_size,
            bias=False
        )

        # Initialize
        self.apply(init_weights)

        # Weight tying
        self.head.weight = self.tok_emb.weight


    def forward(self, idx):
        B, T = idx.shape

        # Positions
        pos = torch.arange(
            T,
            device=idx.device
        )

        # Token + position embeddings
        x = (
            self.tok_emb(idx)
            + self.pos_emb(pos)
        )

        # Fully bidirectional attention.
        # NO MASK.
        for layer in self.layers:
            x = layer(x)

        # Final normalization + output
        x = self.norm(x)

        return self.head(x)


# =====================================================================================================
# MODEL 6 — LexMoE
# =====================================================================================================

class MoEFeedForward(nn.Module):
    """
    Top-1 Mixture-of-Experts feed-forward network.

    Each token is routed to exactly one expert.
    """

    def __init__(
        self,
        d_model,
        d_ff,
        n_experts,
        dropout
    ):
        super().__init__()

        self.n_experts = n_experts

        # Filled during forward()
        self.aux_loss = None
        self.last_choice = None

        # Router
        self.router = nn.Linear(
            d_model,
            n_experts,
            bias=False
        )

        # Experts
        self.experts = nn.ModuleList([
            nn.Sequential(
                nn.Linear(d_model, d_ff),
                nn.GELU(),
                nn.Linear(d_ff, d_model),
                nn.Dropout(dropout),
            )
            for _ in range(n_experts)
        ])


    def forward(self, x):
        B, T, D = x.shape

        # Flatten tokens
        flat = x.reshape(
            B * T,
            D
        )

        # Router probabilities
        router_logits = self.router(flat)

        # Float32 router softmax for numerical stability
        probs = torch.softmax(
            router_logits.float(),
            dim=-1
        )

        # Best expert for each token
        top_p, top_i = probs.max(
            dim=-1
        )

        # Output buffer
        out = flat.new_zeros(
            flat.shape
        )

        # ------------------------------------------------
        # Route tokens to experts
        # ------------------------------------------------

        for e in range(self.n_experts):

            sel = (top_i == e)

            if sel.any():

                expert_out = self.experts[e](
                    flat[sel]
                )

                expert_out = (
                    expert_out
                    * top_p[sel][:, None]
                )

                out[sel] = expert_out.to(
                    out.dtype
                )


        # ------------------------------------------------
        # Switch load-balancing loss
        # ------------------------------------------------

        counts = torch.bincount(
            top_i,
            minlength=self.n_experts
        ).float()

        frac_e = counts / flat.shape[0]

        prob_e = probs.mean(
            dim=0
        )

        self.aux_loss = (
            self.n_experts
            * torch.sum(frac_e * prob_e)
        )


        # Store routing decisions
        self.last_choice = top_i.reshape(
            B,
            T
        )


        # Restore original shape
        return out.reshape(
            B,
            T,
            D
        )


# =====================================================================================================
# MoE BLOCK
# =====================================================================================================

class MoEBlock(nn.Module):
    """
    Pre-norm Transformer block:

        x -> LayerNorm -> Attention -> residual
          -> LayerNorm -> MoE      -> residual
    """

    def __init__(
        self,
        d_model,
        n_heads,
        d_ff,
        n_experts,
        dropout
    ):
        super().__init__()

        # Pre-norm attention
        self.norm1 = nn.LayerNorm(
            d_model
        )

        self.attn = nn.MultiheadAttention(
            d_model,
            n_heads,
            dropout=dropout,
            batch_first=True
        )

        self.drop = nn.Dropout(
            dropout
        )

        # Pre-norm MoE
        self.norm2 = nn.LayerNorm(
            d_model
        )

        self.moe = MoEFeedForward(
            d_model,
            d_ff,
            n_experts,
            dropout
        )


    def forward(self, x, mask):

        # ------------------------------------------------
        # Attention
        # ------------------------------------------------

        h = self.norm1(x)

        attn_out, _ = self.attn(
            h,
            h,
            h,
            attn_mask=mask,
            need_weights=False
        )

        x = x + self.drop(
            attn_out
        )


        # ------------------------------------------------
        # MoE feed-forward
        # ------------------------------------------------

        moe_out = self.moe(
            self.norm2(x)
        )

        x = x + moe_out

        return x


# =====================================================================================================
# LexMoE
# =====================================================================================================

class LexMoE(nn.Module):
    kind = "causal"

    def __init__(
        self,
        vocab_size,
        d_model,
        n_layers,
        n_heads,
        d_ff,
        n_experts,
        max_len,
        dropout=0.1
    ):
        super().__init__()

        self.max_len = max_len

        # Filled during forward()
        self.aux_loss = None

        # Token embedding
        self.tok_emb = nn.Embedding(
            vocab_size,
            d_model
        )

        # Learned positional embedding
        self.pos_emb = nn.Embedding(
            max_len,
            d_model
        )

        # MoE Transformer blocks
        self.blocks = nn.ModuleList([
            MoEBlock(
                d_model,
                n_heads,
                d_ff,
                n_experts,
                dropout
            )
            for _ in range(n_layers)
        ])

        # Final normalization
        self.norm = nn.LayerNorm(
            d_model
        )

        # Output head
        self.head = nn.Linear(
            d_model,
            vocab_size,
            bias=False
        )

        # Initialize
        self.apply(init_weights)

        # Weight tying
        self.head.weight = self.tok_emb.weight


    def forward(self, idx):
        B, T = idx.shape

        # Positions
        pos = torch.arange(
            T,
            device=idx.device
        )

        # Token + position embeddings
        x = (
            self.tok_emb(idx)
            + self.pos_emb(pos)
        )

        # Causal mask
        mask = causal_mask(
            T,
            idx.device
        )

        # Store auxiliary losses
        aux_losses = []

        # Transformer blocks
        for block in self.blocks:

            x = block(
                x,
                mask
            )

            aux_losses.append(
                block.moe.aux_loss
            )

        # Average auxiliary loss across layers
        self.aux_loss = torch.stack(
            aux_losses
        ).mean()

        # Final normalization
        x = self.norm(x)

        # Vocabulary logits
        return self.head(x)


# =====================================================================================================
# FACTORY — DO NOT CHANGE
# =====================================================================================================

def build_model(arch, vocab_size, **cfg):
    classes = {
        "diffusion": LexDiffusionLM,
        "moe": LexMoE
    }

    return classes[arch](
        vocab_size,
        **cfg
    )

## N6. `ss_methods.py` — YOUR CODE (unchanged from Phase 5) + one new method to write
`method_full`, `method_lora`, `method_lorma`, `make_student` and `kd_loss` work as they are: the pipeline hands them
your Phase-1 model as `model.backbone.lm` (with its output layer), and `kd_loss` is applied to the next-token
distributions of the summary tokens (its `-100` mask skips the context). `TokenClassifier` and `kd_loss_multilabel`
are not used this phase. **New — section 8 at the bottom:** `method_prompt` (prompt tuning, on the §12.2.1 list). The
steps are written out there; N7b stops the session if it is still missing and a prompt-tuning run is scheduled.

In [ ]:
%%writefile ss_methods.py
# =====================================================================================================
#  CS-781 Phase 6 (copied from Phase 5) — YOUR CODE: the NER head, the fine-tuning methods, and knowledge distillation
#                                                                AUTHOR: <your name>   (your own code)
# =====================================================================================================

import copy
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils import parametrize
import ss_common as L


# =====================================================================================================
#  1. THE NER HEAD
# =====================================================================================================

class TokenClassifier(nn.Module):

    def __init__(self, backbone, n_tags, dropout):
        super().__init__()

        # Keep backbone
        self.backbone = backbone

        # Dropout before classifier
        self.dropout = nn.Dropout(dropout)

        # Token-level classification head
        self.head = nn.Linear(
            backbone.d_model,
            n_tags
        )


    def forward(self, ids):

        # Backbone hidden states
        h = self.backbone(ids)

        # Classification
        return self.head(
            self.dropout(h)
        )


# =====================================================================================================
#  2. SIMPLE METHODS: full, partial, BitFit
# =====================================================================================================

def method_full(model, **cfg):
    """Full fine-tuning: every parameter of the backbone trains."""

    for p in model.backbone.parameters():
        p.requires_grad_(True)


def method_partial(model, n_last=2, **cfg):
    """
    Partial / layer-wise fine-tuning:
    only the LAST n_last transformer layers + final LayerNorm train.
    """

    # Get the relevant layer stack
    name, layers, final_norm = L.layer_stack(
        model.backbone
    )

    # Train only the last n_last layers
    for layer in layers[-n_last:]:
        for p in layer.parameters():
            p.requires_grad_(True)

    # Train final LayerNorm
    for p in final_norm.parameters():
        p.requires_grad_(True)


def method_bitfit(model, **cfg):
    """
    BitFit:
    only bias parameters of the backbone train.
    """

    for name, p in model.backbone.named_parameters():

        if name.endswith("bias"):
            p.requires_grad_(True)


# =====================================================================================================
#  3. LoRA
# =====================================================================================================

class LoRAParam(nn.Module):
    """
    Additive low-rank update:

        W' = W + (alpha / r) * B @ A
    """

    def __init__(
        self,
        d_out,
        d_in,
        r,
        alpha
    ):
        super().__init__()

        # A: random initialization
        self.A = nn.Parameter(
            torch.empty(r, d_in)
        )

        nn.init.kaiming_uniform_(
            self.A,
            a=math.sqrt(5)
        )

        # B: zero initialization
        # This guarantees W' = W initially.
        self.B = nn.Parameter(
            torch.zeros(d_out, r)
        )

        # Scaling
        self.scale = alpha / r


    def forward(self, W):

        return W + self.scale * (
            self.B @ self.A
        )


def method_lora(
    model,
    r=8,
    alpha=16,
    skip=("head",),
    **cfg
):

    for name, owner, pname, W in L.iter_adaptable(
        model.backbone,
        skip=tuple(skip)
    ):

        d_out, d_in = W.shape

        parametrize.register_parametrization(
            owner,
            pname,
            LoRAParam(
                d_out,
                d_in,
                r,
                alpha
            )
        )


# =====================================================================================================
#  4. LoRMA
# =====================================================================================================

class LoRMAParam(nn.Module):
    """
    LoRMA+ multiplicative low-rank update:

        W' = W + (alpha / r) * B @ (A @ W)

    where:

        A : (r, d_out)
        B : (d_out, r)
    """

    def __init__(
        self,
        d_out,
        r,
        alpha
    ):
        super().__init__()

        # A: random initialization
        self.A = nn.Parameter(
            torch.randn(
                r,
                d_out
            ) * 0.02
        )

        # B: zero initialization
        self.B = nn.Parameter(
            torch.zeros(
                d_out,
                r
            )
        )

        # Scaling
        self.scale = alpha / r


    def forward(self, W):

        # First compute A @ W
        AW = self.A @ W

        # Then B @ (A @ W)
        update = self.B @ AW

        return W + self.scale * update


def method_lorma(
    model,
    r=8,
    alpha=16,
    skip=("head",),
    **cfg
):

    for name, owner, pname, W in L.iter_adaptable(
        model.backbone,
        skip=tuple(skip)
    ):

        d_out, d_in = W.shape

        parametrize.register_parametrization(
            owner,
            pname,
            LoRMAParam(
                d_out,
                r,
                alpha
            )
        )


# =====================================================================================================
#  5. KNOWLEDGE DISTILLATION
# =====================================================================================================

def kd_loss(
    student_logits,
    teacher_logits,
    labels,
    temperature=2.0,
    alpha=0.5
):
    """
    Combined hard-label cross entropy and
    teacher-student KL divergence.
    """

    # Only real labelled tokens
    mask = labels != -100

    s = student_logits[mask]
    t = teacher_logits[mask]
    y = labels[mask]

    # --------------------------------------------------------
    # Hard-label loss
    # --------------------------------------------------------

    hard = F.cross_entropy(
        s,
        y
    )

    # --------------------------------------------------------
    # Soft teacher/student distributions
    # --------------------------------------------------------

    soft = F.kl_div(
        F.log_softmax(
            s / temperature,
            dim=-1
        ),
        F.softmax(
            t / temperature,
            dim=-1
        ),
        reduction="batchmean"
    )

    # Hinton temperature scaling
    soft = soft * (
        temperature * temperature
    )

    # Combined loss
    return (
        alpha * hard
        + (1.0 - alpha) * soft
    )


def make_student(
    backbone,
    keep_layers
):
    """
    Make a smaller copy of a Phase-1 backbone,
    keeping only its first keep_layers layers.
    """

    # Deep copy so original backbone is untouched
    small = copy.deepcopy(
        backbone
    )

    # Find the layer stack
    name, layers, _ = L.layer_stack(
        small
    )

    # Keep only first keep_layers
    setattr(
        small.lm,
        name,
        nn.ModuleList(
            list(layers)[:keep_layers]
        )
    )

    return small


# =====================================================================================================
#  METHOD TABLE
# =====================================================================================================

METHODS = {
    "full": method_full,
    "partial": method_partial,
    "bitfit": method_bitfit,
    "lora": method_lora,
    "lorma": method_lorma,
}


# =====================================================================================================
#  6. YOUR PHASE-3 ADDITIONS (rsLoRA, IA³) — unchanged. Note for Phase 4: the §10.2.1 list is Full fine-tuning,
#     Partial/Layer-wise fine-tuning, Layer freezing/unfreezing, BitFit, LORMA, AdaLoRA, Adapters, Knowledge
#     Distillation. rsLoRA, IA³ and LoRA are NOT on it, so they are optional extras this phase. You already have
#     full, partial (method_partial above), BitFit, LoRMA and distillation — no new method is needed.
# =====================================================================================================

def method_rslora(model, r=8, alpha=16, skip=("head",), **cfg):
    """Rank-stabilised LoRA: use alpha / sqrt(r) as the update scale."""
    if r <= 0:
        raise ValueError("r must be a positive integer")

    for name, owner, pname, W in L.iter_adaptable(
        model.backbone,
        skip=tuple(skip),
    ):
        d_out, d_in = W.shape
        adapter = LoRAParam(d_out, d_in, r, alpha)
        adapter.scale = alpha / math.sqrt(r)
        parametrize.register_parametrization(owner, pname, adapter)


# -----------------------------------------------------------------------------------------------------
#  B. IA³ — "Infused Adapter by Inhibiting and Amplifying Inner Activations" (Liu et al. 2022)
# -----------------------------------------------------------------------------------------------------
#  In plain English:
#    IA³ adds no matrices at all. Every adapted weight gets ONE learned vector, with one number per output
#    row of the weight. Each row of the weight is multiplied by its number. The vector starts as all ones, so
#    at the start the model is unchanged (the same idea as B starting at zero in LoRA). Only these vectors and
#    the head are trained, so IA³ trains far fewer parameters than LoRA.
#
#  What you have to write — two pieces:
#
#    Piece 1 — a small class, written just above method_ia3, shaped like your LoRAParam:
#      1. It inherits from nn.Module and its __init__ receives d_out (the number of output rows of the weight).
#      2. In __init__, after super().__init__(), create one trainable vector of length d_out filled with ones
#         (an nn.Parameter wrapped around torch.ones of size d_out). Call it something like "l".
#      3. Its forward receives the original weight W (shape d_out x d_in) and returns W with every row multiplied
#         by the matching number of the vector. To do that, turn the vector into a column of shape d_out x 1
#         (index it with [:, None]) and multiply it with W — PyTorch repeats the column across all d_in columns.
#
#    Piece 2 — the function method_ia3:
#      4. Go through every adaptable weight with the same loop as method_lora.
#      5. Read the weight's number of output rows (the first number of W.shape).
#      6. Create your Piece-1 object with that number and attach it with parametrize.register_parametrization,
#         exactly like method_lora attaches LoRAParam.
#      7. Delete the "raise NotImplementedError" line.
#
#    For your report: the paper rescales only keys, values and the feed-forward hidden layer. Rescaling every
#    adaptable weight (what steps 4-6 do) is a simpler variant — say which one you used.

class IA3Param(nn.Module):
    """Row-wise IA³ scaling for a weight matrix."""

    def __init__(self, d_out):
        super().__init__()
        self.l = nn.Parameter(torch.ones(d_out))

    def forward(self, W):
        return self.l[:, None] * W


def method_ia3(model, skip=("head",), **cfg):
    """Apply trainable row-wise scaling to every adaptable backbone weight."""
    for name, owner, pname, W in L.iter_adaptable(
        model.backbone,
        skip=tuple(skip),
    ):
        d_out = W.shape[0]
        parametrize.register_parametrization(
            owner,
            pname,
            IA3Param(d_out),
        )


# Do not change this line: it makes "rslora" and "ia3" usable as "method" in N3.
METHODS.update({"rslora": method_rslora, "ia3": method_ia3})


# =====================================================================================================
#  7. NEW FOR PHASE 5 — distillation loss for MULTI-LABEL outputs. YOU write this function (about 6 lines).
# =====================================================================================================
#  Why a new one: your kd_loss above is for single-label classification — a softmax over the classes, and
#  cross_entropy against ONE class index per token. In LSI every statute is its own yes/no decision (a document can
#  have several), so the teacher's knowledge is K independent probabilities, one sigmoid per statute.
#
#  What train_ls.py passes in (already pooled to ONE row per document):
#    student_logits  (B, K)  the student's statute scores        teacher_logits (B, K)  the teacher's
#    targets         (B, K)  the true statutes, 1.0 / 0.0         temperature, alpha     from "kd" in N3
#
#  Steps:
#    1. hard loss: F.binary_cross_entropy_with_logits(student_logits, targets)
#    2. the teacher's soft targets: torch.sigmoid(teacher_logits / temperature)
#    3. soft loss: F.binary_cross_entropy_with_logits(student_logits / temperature, <the soft targets of step 2>),
#       multiplied by temperature * temperature (the same Hinton scaling as in your kd_loss)
#    4. return alpha * hard + (1 - alpha) * soft
#    5. delete the "raise NotImplementedError" line.
#  For your report: this is the per-statute (binary) form of Hinton et al.'s distillation.

def kd_loss_multilabel(student_logits, teacher_logits, targets, temperature=2.0, alpha=0.5):
    hard = F.binary_cross_entropy_with_logits(student_logits, targets)
    soft_targets = torch.sigmoid(teacher_logits.detach() / temperature)
    soft = F.binary_cross_entropy_with_logits(
        student_logits / temperature, soft_targets
    ) * (temperature * temperature)
    return alpha * hard + (1.0 - alpha) * soft


# =====================================================================================================
#  8. NEW FOR PHASE 6 — Prompt Tuning (Lester et al. 2021). YOU write this (about 15 lines).
# =====================================================================================================
#  Why: the §12.2.1 list is Houlsby / Pfeiffer Adapters, Prefix Tuning, Prompt Tuning, LoRA, QLoRA, AdaLoRA, LoRMA,
#  Knowledge Distillation. You already have LoRA, LoRMA and distillation; you need ONE more listed technique.
#  Full fine-tuning, partial, BitFit, rsLoRA and IA³ are not on the list.
#
#  The idea: the whole Phase-1 model stays frozen. Only n_tokens learned vectors (a "soft prompt", d_model numbers
#  each) are trained. They are put in front of every input, and the model learns to read them as an instruction
#  ("summarise this").
#
#  What the pipeline already does for you: when an experiment's method is "prompt", every input sequence (encdec:
#  the encoder input) starts with n_tokens <pad> placeholder tokens. Your job is to swap the embeddings of those
#  placeholders for your learned vectors. A forward hook on the token-embedding layer does that: PyTorch calls
#  hook(module, inputs, output) after every tok_emb(ids) call, and if the hook returns a tensor, it replaces the output.
#
#  Piece 1 — class SoftPrompt(nn.Module), written just above method_prompt:
#    1. __init__(self, tok_emb, n_tokens, pad_id): after super().__init__(), pick n_tokens random row numbers of the
#       embedding table (torch.randint(0, tok_emb.num_embeddings, (n_tokens,))) and make
#       self.prompt = nn.Parameter(tok_emb.weight[those rows].detach().clone())       -> shape (n_tokens, d_model)
#       (starting from real word embeddings trains better than random numbers). Also store n_tokens and pad_id.
#    2. def hook(self, module, inputs, output):   inputs[0] = the token ids (B, T); output = their embeddings (B, T, d)
#       a. n = self.n_tokens. If T <= n, return output unchanged (generation feeds one new token at a time).
#       b. rows = (ids[:, :n] == self.pad_id).all(dim=1)   -> which rows start with the placeholders (the encdec
#          decoder's input starts with <s>, so it is left alone)
#       c. if no row is True, return output unchanged.
#       d. output = output.clone(); output[rows, :n] = self.prompt.to(output.dtype); return output
#
#  Piece 2 — method_prompt(model, n_tokens=20, **cfg):
#    3. sp = SoftPrompt(model.backbone.lm.tok_emb, n_tokens, L.PAD)
#    4. model.soft_prompt = sp            (makes it part of the model: its parameter trains and is saved)
#    5. model.backbone.lm.tok_emb.register_forward_hook(sp.hook)
#    6. delete the "raise NotImplementedError" line.
#  Everything else stays frozen (the training script freezes the model before calling your method).
#  For your report: Lester et al. used ~20-100 prompt tokens on much larger models; say how many you used.

class SoftPrompt(nn.Module):
    """Learnable soft-prompt embeddings inserted at the start of input sequences."""

    def __init__(self, tok_emb, n_tokens, pad_id):
        super().__init__()

        if n_tokens <= 0:
            raise ValueError("n_tokens must be a positive integer")

        # Initialise from existing token embeddings rather than random values.
        rows = torch.randint(
            low=0,
            high=tok_emb.num_embeddings,
            size=(n_tokens,)
        )
        self.prompt = nn.Parameter(
            tok_emb.weight[rows].detach().clone()
        )
        self.n_tokens = n_tokens
        self.pad_id = pad_id

    def hook(self, module, inputs, output):
        # Generation may call the embedding layer with only one new token.
        n = self.n_tokens
        if output.shape[1] <= n:
            return output

        ids = inputs[0]
        # Replace only sequences whose first n tokens are all PAD placeholders.
        rows = (ids[:, :n] == self.pad_id).all(dim=1)
        if not rows.any():
            return output

        output = output.clone()
        prompt = self.prompt.to(device=output.device, dtype=output.dtype)
        output[rows, :n] = prompt
        return output


def method_prompt(model, n_tokens=20, **cfg):
    """Freeze-compatible prompt tuning using learned input embeddings."""
    sp = SoftPrompt(
        model.backbone.lm.tok_emb,
        n_tokens,
        L.PAD
    )
    # Register the prompt as part of the model so it is optimised and saved.
    model.soft_prompt = sp
    # Replace the PAD placeholder embeddings on each token-embedding lookup.
    model.backbone.lm.tok_emb.register_forward_hook(sp.hook)


# Do not change this line: it makes "prompt" usable as "method" in N3.
METHODS.update({"prompt": method_prompt})

## N7. Scripts: training (one experiment), generation (one run), the lab, the official Phase-6 evaluator

In [ ]:
%%writefile train_ss.py
"""Runs ONE experiment from run_config_ss.json: your Phase-1 language model (its own output layer, no new head) + your
fine-tuning method, trained to write the summary after the document context (loss on summary tokens only). Evaluates
the summary-token loss on the validation documents every eval_minutes, keeps the best checkpoint, stops at
time_budget_min or after `epochs`. Generation (and the many zips) happen afterwards in gen_ss.py / the lab.
   python train_ss.py --exp NAME --device cuda:0        (real run)
   python train_ss.py --exp NAME --device cpu --smoke   (tiny random model, few documents: checks the code)"""
import argparse, json, os, random, time
import numpy as np, torch, torch.nn.functional as F
import ss_common as L
import ss_methods as M

ap = argparse.ArgumentParser()
ap.add_argument("--exp", required=True); ap.add_argument("--device", default="cuda:0")
ap.add_argument("--smoke", action="store_true")
args = ap.parse_args()
dev = args.device
cuda = dev.startswith("cuda")
if cuda:
    torch.cuda.set_device(dev)

C = json.load(open(f"{L.WORK}/run_config_ss.json"))
def exp_cfg(name):
    if name in C["experiments"]:
        return {**C["defaults"], **C["experiments"][name], "name": name}
    raise KeyError(f"experiment '{name}' is not in EXPERIMENTS (cell N3)")
E = exp_cfg(args.exp)
RUNS = f"{L.WORK}/{'ss_runs_smoke' if args.smoke else 'ss_runs'}"

def best_epoch_of(name):
    p = f"{RUNS}/{name}/summary.json"
    if not os.path.exists(p):
        p = f"{L.find_runs('summary.json')[name]}/summary.json"
    return json.load(open(p))["best_epoch"]

if E["train_on_all"] and E.get("epochs_from") and not args.smoke:
    E["epochs"] = best_epoch_of(E["epochs_from"])
if args.smoke:
    E.update(epochs=1, batch_size=2, time_budget_min=3, eval_minutes=99, eval_docs=4, sum_tokens=40)
OUT = f"{RUNS}/{args.exp}"; os.makedirs(OUT, exist_ok=True)
log = open(f"{OUT}/log.jsonl", "w")
t0 = time.time()
def say(**kw):
    kw["t"] = round(time.time() - t0); log.write(json.dumps(kw) + "\n"); log.flush()
    print(args.exp, json.dumps(kw), flush=True)
seed = E.get("seed", C["seed"]); random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

# ------------------------------------------------------------------ model
def n_prompt_of(e):
    return int(e["method_cfg"].get("n_tokens", 20)) if e["method"] == "prompt" else 0

def build(e, smoke):
    holder = L.load_holder(e["arch"], C["checkpoints"][e["arch"]], smoke=smoke)
    if e.get("student_layers"):
        holder = M.make_student(holder, e["student_layers"])
    model = L.SumModel(holder)
    for p in model.parameters():
        p.requires_grad_(False)
    M.METHODS[e["method"]](model, **e["method_cfg"])
    return model

model = build(E, args.smoke).to(dev)
lm = model.backbone.lm
fmt = L.fmt_of(lm)
n_prompt = n_prompt_of(E)
teacher = None
if E.get("teacher"):
    TE = exp_cfg(E["teacher"])
    tpath = f"{RUNS}/{E['teacher']}/best.pt"
    if not os.path.exists(tpath):
        tpath = f"{L.find_runs('best.pt')[E['teacher']]}/best.pt"
    teacher = build(TE, args.smoke)
    teacher.load_state_dict(torch.load(tpath, map_location="cpu", weights_only=False)["state"])
    teacher.to(dev).eval()
    for p in teacher.parameters():
        p.requires_grad_(False)
    assert L.fmt_of(teacher.backbone.lm) == fmt and n_prompt_of(TE) == n_prompt and \
        teacher.backbone.lm.max_len == lm.max_len, \
        "teacher and student must read the same sequences: same format (lm / encdec), max_len and prompt length"
ctx_budget, S = L.budgets(lm, n_prompt, E["sum_tokens"], E["ctx_tokens"])
total, trainable = L.count_params(model)

# ------------------------------------------------------------------ data (tokenised once in N8)
train, val = L.Split("train"), L.Split("val")
train_idx = [i for i in range(len(train)) if train.sum_off[i + 1] > train.sum_off[i]]
if E["train_on_all"]:
    sources = [(train, i) for i in train_idx] + [(val, i) for i in range(len(val))]
else:
    sources = [(train, i) for i in train_idx]
if args.smoke:
    sources = sources[:8]
rng = np.random.RandomState(seed)
eval_idx = list(range(len(val)))[:E["eval_docs"]]

def make(split, i, how):
    ctx = L.select_context(split, i, ctx_budget, how)
    return (L.lm_example if fmt == "lm" else L.ed_example)(ctx, split.summary(i), S, n_prompt)

def train_how(r):
    """context strategy of one training example: train_context, or a random one of train_context_mix ("mix")"""
    return E["train_context"] if E["train_context"] != "mix" else E["train_context_mix"][r.randint(len(E["train_context_mix"]))]

say(event="start", arch=E["arch"], method=E["method"], format=fmt, params_total=total, params_trainable=trainable,
    trainable_pct=round(100 * trainable / total, 3), ctx_tokens=ctx_budget, sum_tokens=S, prompt_tokens=n_prompt,
    train_context=E["train_context"], train_docs=len(sources), val_docs=len(eval_idx),
    time_budget_min=E["time_budget_min"], teacher=E.get("teacher"))

# ------------------------------------------------------------------ training setup
params = [p for p in model.parameters() if p.requires_grad]
opt = torch.optim.AdamW(params, lr=E["lr"], weight_decay=E["weight_decay"])
steps_per_epoch = max(1, int(np.ceil(len(sources) / E["batch_size"])))
budget_s = E["time_budget_min"] * 60
total_steps = max(1, int(steps_per_epoch * E["epochs"]))
warm = max(1, int(E["warmup_frac"] * min(total_steps, steps_per_epoch)))
def lr_at(step):            # warm-up, then linear decay to 10% over the planned steps or the time budget, whichever ends first
    frac = max(step / total_steps, (time.time() - t0) / budget_s)
    return E["lr"] * ((step + 1) / warm if step < warm else max(0.1, 1.0 - frac))
scaler = torch.amp.GradScaler("cuda", enabled=cuda)

def batch_loss(batch, net):
    return L.seq_logits(net, fmt, batch, dev)

@torch.no_grad()
def val_loss():
    """mean summary-token cross-entropy on the validation documents (contexts: the first of train_context_mix for
    "mix", else train_context)"""
    model.eval()
    how = E["train_context"] if E["train_context"] != "mix" else E["train_context_mix"][0]
    ex = [make(val, i, how) for i in eval_idx]
    tot, n = 0.0, 0
    for b in L.group_batches([len(e[0]) for e in ex], E["batch_size"], False, None):
        with torch.autocast("cuda", dtype=torch.float16, enabled=cuda):
            logits, y = batch_loss([ex[j] for j in b], model)
        l = F.cross_entropy(logits.float().reshape(-1, logits.size(-1)), y.reshape(-1), ignore_index=L.IGNORE, reduction="sum")
        tot += l.item(); n += int((y != L.IGNORE).sum())
    model.train()
    return tot / max(1, n)

# ------------------------------------------------------------------ train
best, best_epoch, step, last_eval, stop = float("inf"), 0.0, 0, time.time(), False
model.train()
if cuda:
    torch.cuda.reset_peak_memory_stats()
epoch = 0
while not stop:
    epoch += 1
    losses = []
    ex_rng = np.random.RandomState(seed + epoch)
    hows = [train_how(ex_rng) for _ in sources]
    keys = [len(L.select_context(s, i, ctx_budget, h)) for (s, i), h in zip(sources, hows)]
    for bi, b in enumerate(L.group_batches(keys, E["batch_size"], True, rng)):
        if (args.smoke and bi >= 4) or step >= total_steps or time.time() - t0 > budget_s:
            stop = True
            break
        batch = [make(*sources[j], hows[j]) for j in b]
        for g in opt.param_groups:
            g["lr"] = lr_at(step)
        with torch.autocast("cuda", dtype=torch.float16, enabled=cuda):
            logits, y = batch_loss(batch, model)
            logits = logits.float()
            if teacher is not None:
                with torch.no_grad():
                    t_logits = batch_loss(batch, teacher)[0].float()
                loss = M.kd_loss(logits.reshape(-1, logits.size(-1)), t_logits.reshape(-1, t_logits.size(-1)),
                                 y.reshape(-1), **E["kd"])
            else:
                loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), y.reshape(-1), ignore_index=L.IGNORE,
                                       label_smoothing=E["label_smoothing"])
            aux = getattr(lm, "aux_loss", None)
            if E["moe_aux_coef"] and aux is not None:
                loss = loss + E["moe_aux_coef"] * aux
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(params, E["grad_clip"])
        scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
        losses.append(loss.item()); step += 1
        if not E["train_on_all"] and time.time() - last_eval > E["eval_minutes"] * 60:
            vl = val_loss(); last_eval = time.time()
            ep = round(step / steps_per_epoch, 3)
            say(event="eval", epoch=ep, step=step, train_loss=round(float(np.mean(losses[-200:])), 4), val_loss=round(vl, 4))
            if vl < best:
                best, best_epoch = vl, ep
                torch.save({"state": model.state_dict(), "exp": E, "epoch": ep}, f"{OUT}/best.pt")
    if step >= total_steps:
        stop = True
    ep = round(step / steps_per_epoch, 3)
    if losses:
        say(event="epoch_end" if not stop else "train_end", epoch=ep, step=step, train_loss=round(float(np.mean(losses)), 4))
    if E["train_on_all"]:
        torch.save({"state": model.state_dict(), "exp": E, "epoch": ep}, f"{OUT}/best.pt"); best_epoch = ep
    else:
        vl = val_loss(); last_eval = time.time()
        say(event="eval", epoch=ep, step=step, val_loss=round(vl, 4))
        if vl < best:
            best, best_epoch = vl, ep
            torch.save({"state": model.state_dict(), "exp": E, "epoch": ep}, f"{OUT}/best.pt")

peak = round(torch.cuda.max_memory_allocated() / 1e9, 2) if cuda else None
summary = {"exp": args.exp, "arch": E["arch"], "method": E["method"], "teacher": E.get("teacher"),
           "student_layers": E.get("student_layers"), "format": fmt, "train_context": E["train_context"],
           "ctx_tokens": ctx_budget, "sum_tokens": S, "prompt_tokens": n_prompt, "seed": seed,
           "train_on_all": E["train_on_all"], "best_epoch": best_epoch, "epochs_done": round(step / steps_per_epoch, 3),
           "steps": step, "params_total": total, "params_trainable": trainable, "peak_gpu_gb": peak,
           "val_loss": None if E["train_on_all"] else round(best, 4), "train_minutes": round((time.time() - t0) / 60, 1)}
json.dump(summary, open(f"{OUT}/summary.json", "w"), indent=1)
say(event="done", **summary)

In [ ]:
%%writefile gen_ss.py
"""Generates summaries with ONE finished run (its best checkpoint) for every inference context x decoding variant in
run_config_ss.json["lab"]: the first lab.val_docs validation documents (for tentative scores) and every test document
(for the zips). Context "hier" = hierarchical: the document is cut into chunks, each chunk is summarised in a few
words, and the joined chunk summaries are the context of the final summary. Writes
ss_gen/<run>/<context>__<variant>/{val.json, test.json, meta.json}; finished variants are skipped (resumable).
   python gen_ss.py --run NAME --device cuda:0   [--smoke]"""
import argparse, json, os, time
import numpy as np, torch
import ss_common as L
import ss_methods as M

ap = argparse.ArgumentParser()
ap.add_argument("--run", required=True); ap.add_argument("--device", default="cuda:0")
ap.add_argument("--smoke", action="store_true")
args = ap.parse_args()
dev = args.device
if dev.startswith("cuda"):
    torch.cuda.set_device(dev)
C = json.load(open(f"{L.WORK}/run_config_ss.json"))
LAB = C["lab"]
runs = L.find_runs("best.pt", "ss_runs_smoke" if args.smoke else "ss_runs")
folder = runs[args.run]
ck = torch.load(f"{folder}/best.pt", map_location="cpu", weights_only=False)
E = ck["exp"]
summary = json.load(open(f"{folder}/summary.json")) if os.path.exists(f"{folder}/summary.json") else {}

holder = L.load_holder(E["arch"], C["checkpoints"][E["arch"]], smoke=args.smoke)
if E.get("student_layers"):
    holder = M.make_student(holder, E["student_layers"])
model = L.SumModel(holder)
M.METHODS[E["method"]](model, **E["method_cfg"])
model.load_state_dict(ck["state"])
L.merge_adapters(model)
model.to(dev).eval()
lm = model.backbone.lm
n_prompt = int(E["method_cfg"].get("n_tokens", 20)) if E["method"] == "prompt" else 0
ctx_budget, S = L.budgets(lm, n_prompt, E["sum_tokens"], E["ctx_tokens"])

val, test = L.Split("val"), L.Split("test")
val_idx = list(range(min(len(val), LAB["val_docs"])))
test_idx = list(range(len(test)))
if args.smoke:
    val_idx, test_idx = val_idx[:6], test_idx

OUT = f"{L.WORK}/{'ss_gen_smoke' if args.smoke else 'ss_gen'}/{args.run}"
os.makedirs(OUT, exist_ok=True)
print(f"{args.run}: {E['arch']} {E['method']} | context {ctx_budget} tokens, summary up to {S} tokens | "
      f"{len(val_idx)} val + {len(test_idx)} test documents", flush=True)

def contexts(split, idx, how, gen_kw):
    if how != "hier":
        return [L.select_context(split, i, ctx_budget, how) for i in idx]
    # hierarchical: up to H["chunks"] chunks spread over the document -> a short summary of each -> joined
    H = LAB["hier"]
    chunks, owner = [], []
    for j, i in enumerate(idx):
        t = split.tokens(i)
        n = max(1, min(H["chunks"], int(np.ceil(len(t) / ctx_budget))))
        starts = np.linspace(0, max(0, len(t) - ctx_budget), n).astype(int)
        for s in starts:
            c = t[s:s + ctx_budget]
            chunks.append(c[:len(c) - len(c) % L.LEN_BUCKET] if L.LEN_BUCKET <= len(c) < ctx_budget else c)
            owner.append(j)
    kw = {**gen_kw, "max_words": H["chunk_words"], "min_words": min(gen_kw.get("min_words", 0), H["chunk_words"] // 2)}
    parts = L.generate(model, chunks, n_prompt, S, dev, batch_size=LAB["batch_size"], **kw)
    enc = L.bpe_tok.FastEncoder(L.bpe_tok_bpe())
    out = []
    for j in range(len(idx)):
        text = " ".join(p for p, o in zip(parts, owner) if o == j)
        ids = np.asarray(enc.encode(text), dtype=np.int32)[:ctx_budget] if text.strip() else split.tokens(idx[j])[:ctx_budget]
        out.append(ids[:len(ids) - len(ids) % L.LEN_BUCKET] if L.LEN_BUCKET <= len(ids) < ctx_budget else ids)
    return out

for how in LAB["contexts"]:
    for vname, v in LAB["variants"].items():
        d = f"{OUT}/{how}__{vname}"
        if os.path.exists(f"{d}/meta.json"):
            print(f"  {how}__{vname}: done earlier, skipped", flush=True)
            continue
        os.makedirs(d, exist_ok=True)
        t0 = time.time()
        res = {}
        for name, split, idx in (("val", val, val_idx), ("test", test, test_idx)):
            ctx = contexts(split, idx, how, v)
            texts = L.generate(model, ctx, n_prompt, S, dev, batch_size=LAB["batch_size"], **v)
            res[name] = {split.ids[i]: t for i, t in zip(idx, texts)}
            json.dump(res[name], open(f"{d}/{name}.json", "w", encoding="utf-8"), ensure_ascii=False)
        sec = time.time() - t0
        json.dump({"run": args.run, "context": how, "variant": vname, "gen": v, "seconds": round(sec, 1),
                   "sec_per_doc": round(sec / max(1, len(val_idx) + len(test_idx)), 3),
                   "arch": E["arch"], "method": E["method"], "teacher": E.get("teacher"),
                   "params_total": summary.get("params_total"), "params_trainable": summary.get("params_trainable"),
                   "val_loss": summary.get("val_loss")}, open(f"{d}/meta.json", "w"))
        words = np.mean([len(t.split()) for t in res["val"].values()])
        print(f"  {how}__{vname}: {sec:.0f}s, mean {words:.0f} words", flush=True)
print("done", flush=True)

In [ ]:
%%writefile ss_lab.py
"""Phase 6 lab — no training, no generation. Reads every generated variant (ss_gen/<run>/<context>__<variant>/), scores
its validation summaries against the reference summaries with the official metrics (BERTScore F1 with the same
default model as evaluate_phase6.py, ROUGE-1 F1, ROUGE-L F1 with the evaluator's own LCS), adds length / repetition /
abstractiveness statistics, and writes a Codabench zip for the best variants, submissions/index.csv and a score chart.
Tentative score = BERTScore F1 on the held-out validation documents (they were never trained on). You choose which
zips to upload."""
import csv, glob, hashlib, json, os
import numpy as np
import ss_common as L

ARCH_COLORS = {"decoder": "#2a78d6", "prefix": "#eb6834", "encdec": "#1baf7a", "swa": "#eda100", "moe": "#e87ba4"}
METRIC_GREYS = ("#3d3c3a", "#8f8d88", "#c9c7c1")      # right panel: one meaning per colour, so no arch hues there


# ---------------------------------------------------------------- official metric definitions (evaluate_phase6.py)
def _lcs(a, b):
    if len(a) > len(b):
        a, b = b, a
    masks = {}
    for i, t in enumerate(a):
        masks[t] = masks.get(t, 0) | (1 << i)
    s = 0
    for t in b:
        m = masks.get(t, 0); x = m | s; y = (s << 1) | 1; s = x & ~(x - y)
    return bin(s).count("1")


class Scorer:
    """ROUGE-1 / ROUGE-L exactly as evaluate_phase6.py; BERTScore through bert_score with its default English model
    (as bert_score.score(..., lang='en') in the evaluator), loaded once. No bert_score (CPU smoke test) -> None."""
    def __init__(self, use_bert=True):
        from rouge_score import tokenizers, rouge_scorer
        self.r1 = rouge_scorer.RougeScorer(["rouge1"], use_stemmer=True)
        self.tok = tokenizers.DefaultTokenizer(use_stemmer=True)
        self.bert = None
        if use_bert:
            import torch, bert_score
            self.bert = bert_score.BERTScorer(lang="en", device="cuda" if torch.cuda.is_available() else "cpu")

    def rouge(self, preds, refs):
        r1 = rl = 0.0
        for pr, rf in zip(preds, refs):
            r1 += self.r1.score(rf, pr)["rouge1"].fmeasure
            pt, rt = self.tok.tokenize(pr), self.tok.tokenize(rf)
            if not pt and not rt:
                x = 1.0
            elif not pt or not rt:
                x = 0.0
            else:
                z = _lcs(pt, rt); pp, rr = z / len(pt), z / len(rt); x = 0.0 if pp + rr == 0 else 2 * pp * rr / (pp + rr)
            rl += x
        return r1 / len(preds), rl / len(preds)

    def bertscore(self, preds, refs, batch_size=32):
        if self.bert is None:
            return None
        _, _, f = self.bert.score(preds, refs, batch_size=batch_size)
        return float(f.mean())


def rep3(text):
    """% of the summary's word trigrams that repeat an earlier trigram of the same summary (redundancy)"""
    w = text.lower().split()
    g = [tuple(w[i:i + 3]) for i in range(len(w) - 2)]
    return 100.0 * (len(g) - len(set(g))) / len(g) if g else 0.0


def novel2(text, doc_bigrams):
    """% of the summary's word bigrams that do not occur in the document (abstractiveness)"""
    w = text.lower().split()
    g = [tuple(w[i:i + 2]) for i in range(len(w) - 1)]
    return 100.0 * sum(x not in doc_bigrams for x in g) / len(g) if g else 0.0


class Lab:
    def __init__(self, out_dir=f"{L.WORK}/submissions", use_bert=True, sub="ss_gen"):
        self.val, self.test = L.Split("val"), L.Split("test")
        self.ref = dict(zip(self.val.ids, self.val.refs))
        self.vrow = {i: r for r, i in enumerate(self.val.ids)}
        self.scorer = Scorer(use_bert)
        self.out_dir, self.sub = out_dir, sub
        os.makedirs(f"{out_dir}/val_predictions", exist_ok=True)
        self.rows, self._bigrams = [], {}

    def doc_bigrams(self, i):
        if i not in self._bigrams:
            w = L.decode(self.val.tokens(self.vrow[i]).tolist()).lower().split()
            self._bigrams[i] = set(zip(w, w[1:]))
        return self._bigrams[i]

    def collect(self, overrides=None):
        """every finished variant: .../ss_gen/<run>/<context>__<variant>/meta.json (working dir first, then inputs)"""
        found = {}
        for rank, root in enumerate((L.WORK, L.INPUT_ROOT)):
            for p in glob.glob(f"{root}/**/{self.sub}/*/*/meta.json", recursive=True):
                d = os.path.dirname(p)
                key = (os.path.basename(os.path.dirname(d)), os.path.basename(d))
                if key not in found or (rank, -os.path.getmtime(p)) < found[key][0]:
                    found[key] = ((rank, -os.path.getmtime(p)), d)
        dirs = [d for _, d in sorted(found.values(), key=lambda x: x[1])]
        print(f"generated variants found: {len(dirs)} from {len({os.path.dirname(d) for d in dirs})} runs")
        return dirs

    def score_dir(self, d):
        meta = json.load(open(f"{d}/meta.json"))
        vp = json.load(open(f"{d}/val.json", encoding="utf-8"))
        tp = json.load(open(f"{d}/test.json", encoding="utf-8"))
        ids = [i for i in vp if i in self.ref]
        preds = [vp[i].strip() or "the" for i in ids]
        refs = [self.ref[i].strip() for i in ids]
        r1, rl = self.scorer.rouge(preds, refs)
        bs = self.scorer.bertscore(preds, refs)
        tp_ok = all(i in tp for i in self.test.ids)
        row = {"name": f"{meta['run']}__{meta['context']}__{meta['variant']}", "run": meta["run"], "arch": meta["arch"],
               "method": meta["method"] + (f" (KD from {meta['teacher']})" if meta.get("teacher") else ""),
               "context": meta["context"], "decoding": meta["variant"],
               "bertscore_f1": None if bs is None else round(bs, 4), "rouge1": round(r1, 4), "rougeL": round(rl, 4),
               "val_docs": len(ids), "avg_words": round(float(np.mean([len(p.split()) for p in preds])), 1),
               "ref_avg_words": round(float(np.mean([len(r.split()) for r in refs])), 1),
               "rep3_pct": round(float(np.mean([rep3(p) for p in preds])), 2),
               "novel2_pct": round(float(np.mean([novel2(p, self.doc_bigrams(i)) for p, i in zip(preds, ids)])), 2),
               "sec_per_doc": meta.get("sec_per_doc"), "val_loss": meta.get("val_loss"),
               "trainable_pct": round(100 * meta["params_trainable"] / meta["params_total"], 3)
               if meta.get("params_total") else None, "params_total": meta.get("params_total")}
        self.rows.append((row, {i: vp[i] for i in ids}, tp if tp_ok else None))
        if not tp_ok:
            print(f"{row['name']}: test.json misses documents -> no zip")
        return row

    def score_all(self, dirs):
        for k, d in enumerate(dirs):
            r = self.score_dir(d)
            bs = "n/a" if r["bertscore_f1"] is None else f"{r['bertscore_f1']:.4f}"
            print(f"  [{k + 1}/{len(dirs)}] {r['name'][:70]:70s} BERTScore {bs}  R1 {r['rouge1']:.4f}  RL {r['rougeL']:.4f}", flush=True)

    def lead_baseline(self, words=300):
        """reference row only (not a Phase-1 model, never zipped): the first `words` words of each document"""
        ids = list(self.ref)[:max(r[0]["val_docs"] for r in self.rows)] if self.rows else list(self.ref)
        preds = [" ".join(L.decode(self.val.tokens(self.vrow[i])[:words * 2].tolist()).split()[:words]) or "the" for i in ids]
        refs = [self.ref[i].strip() for i in ids]
        r1, rl = self.scorer.rouge(preds, refs)
        bs = self.scorer.bertscore(preds, refs)
        row = {"name": f"(reference) lead-{words} words of the document", "run": "-", "arch": "-", "method": "extractive",
               "context": "head", "decoding": "-", "bertscore_f1": None if bs is None else round(bs, 4),
               "rouge1": round(r1, 4), "rougeL": round(rl, 4), "val_docs": len(ids),
               "avg_words": round(float(np.mean([len(p.split()) for p in preds])), 1)}
        self.rows.append((row, None, None))
        print(f"lead-{words} reference: BERTScore {row['bertscore_f1']}  R1 {row['rouge1']}  RL {row['rougeL']}")

    def finish(self, top_n=40):
        """zips for: the top_n variants by BERTScore (ROUGE-L when BERTScore is unavailable), the best variant of every
        run. Identical test outputs are written once. index.csv lists every variant."""
        key = lambda r: -(r[0]["bertscore_f1"] if r[0]["bertscore_f1"] is not None else r[0]["rougeL"])
        order = sorted(self.rows, key=key)
        zippable = [r for r in order if r[2] is not None]
        keep = {id(r) for r in zippable[:top_n]}
        best_of = {}
        for r in zippable:
            best_of.setdefault(r[0]["run"], r)
        keep |= {id(r) for r in best_of.values()}
        for f in glob.glob(f"{self.out_dir}/*.zip") + glob.glob(f"{self.out_dir}/val_predictions/*__*.json"):
            os.remove(f)
        json.dump({i: self.ref[i] for i in self.ref}, open(f"{self.out_dir}/val_predictions/val_truth_all.json", "w", encoding="utf-8"),
                  ensure_ascii=False)
        seen, n = {}, 0
        for r in order:
            row, vp, tp = r
            row["file"] = ""
            if id(r) not in keep:
                continue
            h = hashlib.md5(json.dumps([tp[i] for i in self.test.ids]).encode()).hexdigest()
            if h in seen:
                row["file"] = f"(same as {seen[h]})"
                continue
            seen[h] = row["file"] = f"{row['name']}.zip"
            L.write_submission(self.test.ids, tp, f"{self.out_dir}/_prediction.json", f"{self.out_dir}/{row['file']}")
            json.dump([{"id": i, "summary": s.strip() or "the"} for i, s in vp.items()],
                      open(f"{self.out_dir}/val_predictions/{row['name']}.json", "w", encoding="utf-8"), ensure_ascii=False)
            n += 1
        if os.path.exists(f"{self.out_dir}/_prediction.json"):
            os.remove(f"{self.out_dir}/_prediction.json")
        cols = ["name", "bertscore_f1", "rouge1", "rougeL", "avg_words", "ref_avg_words", "rep3_pct", "novel2_pct",
                "run", "arch", "method", "context", "decoding", "val_docs", "sec_per_doc", "val_loss", "trainable_pct",
                "params_total", "file"]
        with open(f"{self.out_dir}/index.csv", "w", newline="", encoding="utf-8") as f:
            w = csv.DictWriter(f, fieldnames=cols, extrasaction="ignore")
            w.writeheader()
            for row, _, _ in order:
                w.writerow(row)
        print(f"{len(self.rows)} variants scored, {n} zips written to {self.out_dir}/ (index.csv lists all)")
        return [r[0] for r in order]


def table(rows, n=40):
    print(f"{'BERTSc':>6s} {'R-1':>6s} {'R-L':>6s} {'words':>5s} {'rep3%':>5s} {'nov2%':>5s}  {'variant':66s} zip")
    for r in rows[:n]:
        bs = "  n/a" if r["bertscore_f1"] is None else f"{r['bertscore_f1']:.4f}"
        print(f"{bs:>6s} {r['rouge1']:6.4f} {r['rougeL']:6.4f} {r['avg_words']:5.0f} {r.get('rep3_pct') or 0:5.1f} "
              f"{r.get('novel2_pct') or 0:5.1f}  {r['name'][:66]:66s} {r.get('file', '')}")


def chart(rows, path, n=25):
    """left: the n best variants by tentative BERTScore, coloured by Phase-1 architecture (the extractive lead
    reference in grey outline); right: the best variant of every run with BERTScore, ROUGE-1 and ROUGE-L.
    index.csv is the table view of the same numbers."""
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    metric = "bertscore_f1" if any(r["bertscore_f1"] is not None for r in rows) else "rougeL"
    label = "BERTScore F1" if metric == "bertscore_f1" else "ROUGE-L F1 (BERTScore unavailable)"
    scored = [r for r in rows if r[metric] is not None]
    if not scored:
        print("nothing to plot"); return
    top = scored[:n][::-1]
    best_of = {}
    for r in scored:
        if r["run"] != "-":
            best_of.setdefault(r["run"], r)
    groups = list(best_of.values())[:15][::-1]
    ink, muted, grid, bg = "#0b0b0b", "#52514e", "#e4e3df", "#fcfcfb"
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(17, max(6, 0.32 * max(len(top), 3 * len(groups) / 1.6))),
                                 gridspec_kw={"width_ratios": [1.15, 1]})
    fig.patch.set_facecolor(bg)
    for ax in (a1, a2):
        ax.set_facecolor(bg)
        for s in ("top", "right", "left"):
            ax.spines[s].set_visible(False)
        ax.spines["bottom"].set_color(grid)
        ax.tick_params(colors=muted, labelsize=8, length=0)
        ax.grid(axis="x", color=grid, linewidth=0.8)
        ax.set_axisbelow(True)
    y = np.arange(len(top))
    a1.barh(y, [r[metric] for r in top], height=0.62,
            color=[ARCH_COLORS.get(r["arch"], bg) for r in top],
            edgecolor=[muted if r["arch"] not in ARCH_COLORS else bg for r in top], linewidth=1.2)
    a1.set_yticks(y, [r["name"][:62] for r in top], fontsize=7.5, color=ink)
    for yi, r in zip(y, top):
        a1.text(r[metric] + 0.003, yi, f"{r[metric]:.3f}", va="center", fontsize=7.5, color=muted)
    lo = min(r[metric] for r in top)
    a1.set_xlim(max(0, lo - 0.05) if metric == "bertscore_f1" else 0, min(1.0, max(r[metric] for r in top) * 1.04))
    a1.set_xlabel(f"tentative {label} on the validation documents", color=muted, fontsize=9)
    a1.set_title(f"Top {len(top)} variants (run__context__decoding)", loc="left", color=ink, fontsize=11)
    archs = [a for a in ARCH_COLORS if any(r["arch"] == a for r in top)]
    a1.legend([plt.Rectangle((0, 0), 1, 1, color=ARCH_COLORS[a]) for a in archs], archs, loc="lower right",
              frameon=False, fontsize=8)
    series = [(metric, label.split(" (")[0], METRIC_GREYS[0]), ("rouge1", "ROUGE-1 F1", METRIC_GREYS[1]),
              ("rougeL", "ROUGE-L F1", METRIC_GREYS[2])]
    if metric == "rougeL":
        series = series[1:]
    yy = np.arange(len(groups)); h = 0.8 / len(series)
    for k, (key, lab, col) in enumerate(series):
        a2.barh(yy + (len(series) / 2 - 0.5 - k) * h, [r[key] or 0 for r in groups], height=h, color=col, label=lab,
                edgecolor=bg, linewidth=1.5)
        for yi, r in zip(yy, groups):
            a2.text((r[key] or 0) + 0.004, yi + (len(series) / 2 - 0.5 - k) * h, f"{r[key]:.3f}", va="center",
                    fontsize=7, color=muted)
    a2.set_yticks(yy, [f"{r['run']}  ({r['context']}, {r['decoding']})"[:52] for r in groups], fontsize=7.5, color=ink)
    a2.set_xlim(0, min(1.0, max(max(r[k] or 0 for r in groups) for k, _, _ in series) * 1.15))
    a2.set_xlabel("validation scores of the best variant of each run", color=muted, fontsize=9)
    a2.set_title("Best per run", loc="left", color=ink, fontsize=11)
    a2.legend(loc="lower right", frameon=False, fontsize=8)
    fig.tight_layout()
    fig.savefig(path, dpi=130, facecolor=fig.get_facecolor())
    plt.close(fig)
    print("chart saved:", path)

In [ ]:
%%writefile evaluate_phase6.py
#!/usr/bin/env python3
"""Local evaluator for Phase 6.

Requires: rouge-score, bert-score, torch.
Truth JSON: {"doc_id":"reference summary", ...}
Prediction JSON: either {"doc_id":"summary", ...} or [{"id":"...","summary":"..."}, ...]
"""
import argparse,json,sys
try:
    from rouge_score import tokenizers, rouge_scorer
    import bert_score
except ImportError as e:
    print('Missing dependency. Install requirements.txt first.', file=sys.stderr); raise

def load(path):
    with open(path,encoding='utf-8') as f: return json.load(f)

def lcs(a,b):
    if len(a)>len(b): a,b=b,a
    masks={}
    for i,t in enumerate(a): masks[t]=masks.get(t,0)|(1<<i)
    s=0
    for t in b:
        m=masks.get(t,0); x=m|s; y=(s<<1)|1; s=x & ~(x-y)
    return s.bit_count() if hasattr(int,'bit_count') else bin(s).count('1')

def read_pred(x):
    if isinstance(x,dict): out=x
    elif isinstance(x,list):
        out={}
        for r in x:
            if not isinstance(r,dict) or 'id' not in r or 'summary' not in r: raise ValueError('Each prediction item must contain id and summary.')
            if r['id'] in out: raise ValueError(f'Duplicate prediction ID: {r["id"]}')
            out[r['id']]=r['summary']
    else: raise ValueError('Prediction must be a JSON object or list of objects.')
    for k,v in out.items():
        if not isinstance(k,str) or not isinstance(v,str) or not v.strip(): raise ValueError(f'Invalid summary for ID {k!r}.')
    return out

def main():
    ap=argparse.ArgumentParser(description='Evaluate Phase 6 summaries on your own held-out split.')
    ap.add_argument('--truth',required=True); ap.add_argument('--prediction',required=True); a=ap.parse_args()
    t=load(a.truth); p=read_pred(load(a.prediction))
    if not isinstance(t,dict) or not t: raise ValueError('Truth must be a non-empty JSON object.')
    if set(t)!=set(p): raise ValueError(f"ID mismatch: missing={len(set(t)-set(p))}, extra={len(set(p)-set(t))}")
    preds=[p[k].strip() for k in t]; refs=[t[k].strip() for k in t]
    scorer=rouge_scorer.RougeScorer(['rouge1'],use_stemmer=True); tok=tokenizers.DefaultTokenizer(use_stemmer=True)
    r1=rl=0.0
    for pr,rf in zip(preds,refs):
        s=scorer.score(rf,pr); r1+=s['rouge1'].fmeasure
        pt=tok.tokenize(pr); rt=tok.tokenize(rf)
        if not pt and not rt: x=1.0
        elif not pt or not rt: x=0.0
        else:
            z=lcs(pt,rt); pp=z/len(pt); rr=z/len(rt); x=0.0 if pp+rr==0 else 2*pp*rr/(pp+rr)
        rl+=x
    n=len(t); r1/=n; rl/=n
    try:
        import torch; device='cuda' if torch.cuda.is_available() else 'cpu'
    except ImportError: device=None
    _,_,f=bert_score.score(preds,refs,lang='en',batch_size=8,verbose=False,**({'device':device} if device else {}))
    print(f'Documents: {n}'); print(f'BERTScore F1: {float(f.mean().item()):.6f}'); print(f'ROUGE-1: {r1:.6f}'); print(f'ROUGE-L: {rl:.6f}')
if __name__=='__main__':
    try: main()
    except Exception as e: print(f'EVALUATION FAILED: {e}',file=sys.stderr); sys.exit(1)

## N7b. Pre-flight check — every input present, prompt tuning written? (stops here if not)

In [ ]:
# N7b — PRE-FLIGHT: checks every input this session needs and STOPS here (within a minute, not after hours) if one
# is missing. Read the printout: every line must show a path, not MISSING.
import json, glob, os, importlib, torch
import ss_common as L
C = json.load(open("/kaggle/working/run_config_ss.json"))
problems = []
print("tokenizer:", L.find_input("bpe_tok.py"), "|", L.find_input("bpe.json"))
archs = set()
for e in SESSION_EXPERIMENTS + [r for r in GEN_RUNS if r in C["experiments"]]:
    x = C["experiments"][e]
    archs.add(x["arch"])
    if x.get("teacher"):
        archs.add(C["experiments"][x["teacher"]]["arch"])
for arch in sorted(archs):
    pat = C["checkpoints"][arch]
    hits = sorted(glob.glob(f"{L.INPUT_ROOT}/**/{pat}", recursive=True))
    print(f"Phase-1 checkpoint {arch:8s} ({pat}): {hits[0] if hits else 'MISSING'}")
    if not hits:
        problems.append(f"the Phase-1 '{arch}' checkpoint '{pat}' is not in any input — add the notebook / dataset "
                        f"that contains it" + (" (decoder: notebook27acca3fc3 = notebook D)" if arch == "decoder" else
                                               " (moe: Phase1_training2 / phase1-train-extra)" if arch == "moe" else ""))
for e in SESSION_EXPERIMENTS:
    t = C["experiments"][e].get("teacher")
    if t and t not in SESSION_EXPERIMENTS:
        f = L.find_runs("best.pt").get(t)
        print(f"distillation teacher '{t}' for {e}: {f + '/best.pt' if f else 'MISSING'}")
        if not f:
            problems.append(f"the trained teacher '{t}' was not found — add it to SESSION_EXPERIMENTS")
for r in GEN_RUNS:
    if r not in SESSION_EXPERIMENTS and r not in L.find_runs("best.pt"):
        problems.append(f"GEN_RUNS: '{r}' is neither trained this session nor found in the inputs")
if any(C["experiments"][e]["method"] == "prompt" for e in SESSION_EXPERIMENTS):
    import ss_methods as M
    importlib.reload(M)
    try:
        m = L.SumModel(L.LMHolder("decoder", L.build_lm("decoder", L.TINY["decoder"]))).eval()
        for p in m.parameters():
            p.requires_grad_(False)
        x = torch.tensor([[L.PAD] * 4 + [10, 11, 12, L.BOS, 13]])
        before = m(x).detach()
        M.METHODS["prompt"](m, n_tokens=4)
        after = m(x)
        trainable = [n for n, p in m.named_parameters() if p.requires_grad]
        assert trainable, "method_prompt created no trainable parameter (is model.soft_prompt set?)"
        assert not torch.allclose(before, after.detach()), "the soft prompt does not change the model's output (hook?)"
        print(f"method_prompt: OK (trainable: {trainable})")
    except NotImplementedError:
        problems.append("method_prompt in N6 (section 8) is not written yet — write it, or take the prompt-tuning "
                        "runs out of SESSION_EXPERIMENTS / GEN_RUNS")
    except AssertionError as e:
        problems.append(f"method_prompt: {e}")
try:
    import bert_score, rouge_score
    print("metric packages: OK")
except ImportError as e:
    problems.append(f"metric package missing ({e}) — is Internet on? N1 installs rouge-score and bert-score")
print("finished runs found in the inputs (they join generation / the lab):", sorted(L.find_runs("best.pt")) or "none")
if problems:
    raise RuntimeError("PRE-FLIGHT FAILED — nothing was trained:\n  - " + "\n  - ".join(problems))
print("\nPRE-FLIGHT OK")

## N8. Data check + validation split + one-time tokenisation (sizes, document / summary lengths, budgets)

In [ ]:
# N8 — read the data, set aside the validation split, tokenise everything ONCE (all CPU cores; cached in
# /kaggle/working/p6cache), then show sizes and lengths. If this cell fails, paste the error.
import json, time, importlib, glob, os, shutil, numpy as np
import ss_common as L
importlib.reload(L)
C = json.load(open("/kaggle/working/run_config_ss.json"))
prev = sorted(glob.glob(f"{L.INPUT_ROOT}/**/p6cache/test_meta.json", recursive=True))
if prev and not os.path.exists(L.CACHE):                 # an earlier run's cache: no need to tokenise again
    shutil.copytree(os.path.dirname(prev[0]), L.CACHE)
    print("reusing the tokenisation cache from", os.path.dirname(prev[0]))
t0 = time.time()
L.build_cache(C["val_docs"], seed=C["seed"])
print(f"tokenised in {time.time() - t0:.0f}s")
S = C["defaults"]["sum_tokens"]
for name in L.SPLITS:
    s = L.Split(name)
    d = s.doc_lengths()
    line = f"{name:5s}: {len(s):5d} docs | document tokens median {int(np.median(d))}, 90% {int(np.percentile(d, 90))}, max {d.max()}"
    if name != "test":
        m = s.sum_lengths()
        line += (f" | summary tokens median {int(np.median(m))}, 90% {int(np.percentile(m, 90))}, max {m.max()} | "
                 f"{100 * np.mean(m > S):.0f}% longer than sum_tokens={S} | compression {np.median(d / np.maximum(m, 1)):.1f}x")
    print(line)
val = L.Split("val")
w = [len(r.split()) for r in val.refs]
print(f"\nvalidation reference summaries: words median {int(np.median(w))}, 90% {int(np.percentile(w, 90))}")
print("example reference (first 60 words):", " ".join(val.refs[0].split()[:60]), "...")
test = L.Split("test")
assert len(set(test.ids)) == len(test.ids), "test document ids repeat"
print("test documents:", len(test), "| example id:", test.ids[0])
for arch in sorted({C["experiments"][e]["arch"] for e in SESSION_EXPERIMENTS}):
    try:
        h = L.load_holder(arch, C["checkpoints"][arch])
        e = next(x for x in SESSION_EXPERIMENTS if C["experiments"][x]["arch"] == arch)
        ex = {**C["defaults"], **C["experiments"][e]}
        n_prompt = ex["method_cfg"].get("n_tokens", 20) if ex["method"] == "prompt" else 0
        print(f"{arch:8s}: context {L.budgets(h.lm, n_prompt, ex['sum_tokens'], ex['ctx_tokens'])[0]} tokens, "
              f"summary up to {L.budgets(h.lm, n_prompt, ex['sum_tokens'], ex['ctx_tokens'])[1]} tokens")
    except Exception as e:
        print(f"{arch:8s}: MISSING -> {e}")

## N9. CPU smoke test: training + generation of the session's experiments, and the generation cache check

In [ ]:
# N9 — (a) the key/value-cache generator gives the same next-token scores as your models' own forward pass;
# (b) every experiment trains (tiny random models, a few documents) and generates: each must end with exit code 0.
import json, subprocess, sys, torch, importlib
import ss_common as L
importlib.reload(L)
# The reference is your model's forward in train mode (dropout is 0 in these tiny models): that is the computation
# training and every GPU forward use. In eval mode without autocast PyTorch may switch to its "fast path", which
# treats swa's per-head ALiBi bias differently — on the GPU autocast disables that path, so it never applies there.
for arch in ("decoder", "prefix", "swa", "moe"):
    torch.manual_seed(0)
    lm = L.build_lm(arch, L.TINY[arch]).train()
    x = torch.randint(5, 200, (2, 40))
    with torch.no_grad():
        r = L.CachedLM(lm); out = [r.prefill(x[:, :30])] + [r.step(x[:, k:k + 1]) for k in range(30, 39)]
        ref = [(lm(x[:, :k + 1], prefix_len=30) if arch == "prefix" else lm(x[:, :k + 1]))[:, -1] for k in range(29, 39)]
    err = max((a - b).abs().max().item() for a, b in zip(out, ref))
    print(f"cache check {arch:8s}: max difference {err:.2e} {'OK' if err < 1e-3 else 'PROBLEM'}")
C = json.load(open("/kaggle/working/run_config_ss.json"))
smoke = []
for exp in SMOKE_EXPERIMENTS:
    t = C["experiments"][exp].get("teacher")
    smoke += [x for x in ([t] if t else []) + [exp] if x not in smoke]
for exp in smoke:
    for script, flag in (("train_ss.py", "--exp"), ("gen_ss.py", "--run")):
        r = subprocess.run([sys.executable, script, flag, exp, "--device", "cpu", "--smoke"],
                           capture_output=True, text=True, cwd="/kaggle/working")
        lines = [l for l in (r.stdout + r.stderr).splitlines() if "Warning" not in l and "warn(" not in l]
        print(f"===== {script} {exp}: exit code {r.returncode}")
        print("\n".join(lines[-2:] if r.returncode == 0 else lines[-25:]))

## N10. Train the session's experiments, then generate with every run — on both GPUs

In [ ]:
# N10 — one job per GPU at a time. Jobs: train every SESSION_EXPERIMENT, then generate with every GEN_RUN
# (a run's generation waits for its training; a distillation student waits for its teacher). Progress every 5 minutes.
import subprocess, sys, time, os, json, torch
C = json.load(open("/kaggle/working/run_config_ss.json"))
devices = [f"cuda:{i}" for i in range(torch.cuda.device_count())] or ["cpu"]
pending = [("train", e) for e in SESSION_EXPERIMENTS] + [("gen", r) for r in GEN_RUNS]
running, finished = {}, {}

def ready(job):
    kind, name = job
    if kind == "gen":
        waits = [("train", name)]
    else:
        e = C["experiments"][name]
        waits = [("train", x) for x in (e.get("teacher"), e.get("epochs_from")) if x]
    busy = set(pending) | set(running.values())
    return all(w not in busy for w in waits)

last_print = 0
while pending or running:
    for dev in devices:
        if dev in running and running[dev][2].poll() is not None:
            job = running.pop(dev)
            finished[job[:2]] = job[2].returncode
            print(time.strftime("%H:%M"), f"| {job[0]} {job[1]} finished on {dev} with exit code {job[2].returncode}", flush=True)
        if dev not in running:
            nxt = next((j for j in pending if ready(j)), None)
            if nxt:
                pending.remove(nxt)
                if nxt[0] == "gen" and finished.get(("train", nxt[1]), 0) != 0:
                    print(f"skipping generation of {nxt[1]}: its training failed"); continue
                out = open(f"/kaggle/working/{nxt[0]}_{nxt[1]}.out", "w")
                cmd = ["train_ss.py", "--exp"] if nxt[0] == "train" else ["gen_ss.py", "--run"]
                running[dev] = (*nxt, subprocess.Popen([sys.executable, cmd[0], cmd[1], nxt[1], "--device", dev],
                                                        stdout=out, stderr=subprocess.STDOUT, cwd="/kaggle/working"))
                print(time.strftime("%H:%M"), f"| started {nxt[0]} {nxt[1]} on {dev}", flush=True)
    if time.time() - last_print > 300 and running:
        for dev, (kind, name, _) in running.items():
            lines = [l for l in open(f"/kaggle/working/{kind}_{name}.out").read().splitlines() if l.strip()]
            print(time.strftime("%H:%M"), "|", kind, "|", (lines[-1] if lines else f"{name}: starting...")[:220], flush=True)
        last_print = time.time()
    time.sleep(15)
print("\nexit codes:", {f"{k} {n}": c for (k, n), c in finished.items()} or "nothing ran this session")
for (kind, name), code_ in finished.items():
    if code_ != 0:
        print(f"\n===== {kind} {name} FAILED — last lines:\n" + open(f"/kaggle/working/{kind}_{name}.out").read()[-3000:])

In [ ]:
# N10b — what each run reached (validation summary-token loss of the kept checkpoint, size, time, memory)
import json, os
import ss_common as L
rows = [json.load(open(f"{f}/summary.json")) for e, f in L.find_runs("summary.json").items()]
print(f"{'run':22s} {'arch':8s} {'method':7s} {'train%':>7s} {'val loss':>8s} {'params':>8s} {'ctx':>5s} {'epochs':>7s} {'min':>5s} {'GPU GB':>6s}")
for s in sorted(rows, key=lambda s: s.get("val_loss") or 99):
    print(f"{s['exp']:22s} {s['arch']:8s} {s['method']:7s} {100 * s['params_trainable'] / s['params_total']:7.2f} "
          f"{str(s.get('val_loss')):>8s} {s['params_total'] / 1e6:7.1f}M {s['ctx_tokens']:5d} {s['epochs_done']:7.2f} "
          f"{s['train_minutes']:5.0f} {str(s['peak_gpu_gb']):>6s}")

## N11. The lab — score every generated variant, zips, index.csv, score chart

In [ ]:
# N11 — the lab (GPU for BERTScore, ~10-20 min): scores every generated variant on the validation documents, adds the
# extractive lead reference row, writes zips (TOP_N_ZIPS best + best of each run), index.csv and the score chart.
import importlib, ss_common as L, ss_lab
importlib.reload(L); importlib.reload(ss_lab)
lab = ss_lab.Lab()
lab.score_all(lab.collect())
lab.lead_baseline()
rows = lab.finish(TOP_N_ZIPS)
ss_lab.table(rows, 45)
ss_lab.chart(rows, "/kaggle/working/submissions/score_chart.png")
from IPython.display import Image, display
display(Image("/kaggle/working/submissions/score_chart.png"))

## N12. Check the zips + confirm the lab's scores with the official evaluator

In [ ]:
# N12 — check every zip (format, ids, non-empty summaries) and confirm the lab's scores with the official evaluator
# (evaluate_phase6.py) on the 3 best written variants.
import zipfile, json, glob, csv, subprocess, sys
import ss_common as L
test_ids = L.Split("test").ids
zips = sorted(glob.glob("/kaggle/working/submissions/*.zip"))
bad = 0
for z in zips:
    with zipfile.ZipFile(z) as zf:
        names, pred = zf.namelist(), json.loads(zf.read("prediction.json"))
    ids = [r["id"] for r in pred]
    ok = (names == ["prediction.json"] and isinstance(pred, list) and len(ids) == len(set(ids)) == len(test_ids)
          and set(ids) == set(test_ids) and all(isinstance(r["summary"], str) and r["summary"].strip() for r in pred))
    bad += not ok
    if not ok:
        print("PROBLEM", z)
print(f"{len(zips)} zips checked, {bad} with problems")
V = "/kaggle/working/submissions/val_predictions"
truth_all = json.load(open(f"{V}/val_truth_all.json", encoding="utf-8"))
rows = [r for r in csv.DictReader(open("/kaggle/working/submissions/index.csv", encoding="utf-8")) if r["file"].endswith(".zip")]
for r in rows[:3]:
    pred = json.load(open(f"{V}/{r['name']}.json", encoding="utf-8"))
    json.dump({p["id"]: truth_all[p["id"]] for p in pred}, open(f"{V}/_truth.json", "w", encoding="utf-8"), ensure_ascii=False)
    out = subprocess.run([sys.executable, "evaluate_phase6.py", "--truth", f"{V}/_truth.json", "--prediction",
                          f"{V}/{r['name']}.json"], capture_output=True, text=True)
    got = {l.split(":")[0]: l.split()[-1] for l in out.stdout.splitlines() if ":" in l}
    print(f"{r['name'][:60]:60s} lab BERTScore {r['bertscore_f1']} R1 {r['rouge1']} RL {r['rougeL']} | official "
          f"{got.get('BERTScore F1', out.stderr.strip()[-150:])} {got.get('ROUGE-1', '')} {got.get('ROUGE-L', '')}")

## N13. Material for the analysis: contexts, decoding, length, redundancy; example summaries

In [ ]:
# N13 — tables from index.csv (no conclusions drawn — that part is yours): mean scores per run x context and per run
# x decoding, length / repetition / novelty of each run's best variant, and two summaries of the best variant next
# to their references.
import csv, json, numpy as np
from collections import defaultdict
rows = [r for r in csv.DictReader(open("/kaggle/working/submissions/index.csv", encoding="utf-8")) if r["run"] != "-"]
metric = "bertscore_f1" if rows and rows[0]["bertscore_f1"] else "rougeL"
def pivot(key):
    t = defaultdict(dict)
    for r in rows:
        t[r["run"]].setdefault(r[key], []).append(float(r[metric]))
    cols = sorted({r[key] for r in rows})
    print(f"\nmean {metric} by run x {key}:")
    print(f"{'run':22s} " + " ".join(f"{c[:16]:>16s}" for c in cols))
    for run, d in sorted(t.items()):
        print(f"{run:22s} " + " ".join(f"{np.mean(d[c]):16.4f}" if c in d else f"{'':>16s}" for c in cols))
pivot("context"); pivot("decoding")
print(f"\nbest variant of each run: words (reference {rows[0]['ref_avg_words'] if rows else '?'}), repeated trigrams %, "
      "novel bigrams % (abstractiveness), seconds per document, trainable %:")
seen = set()
for r in rows:
    if r["run"] in seen:
        continue
    seen.add(r["run"])
    print(f"  {r['run']:22s} {metric} {float(r[metric]):.4f} | words {r['avg_words']:>6s} | rep3 {r['rep3_pct']:>5s} | "
          f"novel2 {r['novel2_pct']:>5s} | {r['sec_per_doc']:>6s} s/doc | trainable {r['trainable_pct']}%")
best = next((r for r in rows if r["file"].endswith(".zip")), None)
if best:
    V = "/kaggle/working/submissions/val_predictions"
    pred = json.load(open(f"{V}/{best['name']}.json", encoding="utf-8"))
    truth = json.load(open(f"{V}/val_truth_all.json", encoding="utf-8"))
    for p in pred[:2]:
        print(f"\n===== {best['name']} | {p['id']}\nGENERATED: {' '.join(p['summary'].split()[:120])} ...\n"
              f"REFERENCE: {' '.join(truth[p['id']].split()[:120])} ...")